# Automation and Affordability in U.S. Counties

Is the task composition of local work associated with what residents can afford?

Chris Bell  
Julian Pacheco

In [1]:
import os, sys
os.environ["R_HOME"]=os.path.join(sys.prefix, "lib", "R")
%load_ext rpy2.ipython
import rpy2.robjects as ro
ro.r('''
suppressMessages({
  library(ggplot2)
  library(patchwork)
  library(scales)
})

# Semantic palette. Validated 11 Aug 2026 against the dataviz skill's colorblind-separation
# and contrast checks (all-pairs mode) — see CLAUDE.md for the validated hex values and rationale.
# BLUE substantive result, ORANGE failed specification,
# GREEN corrected or preferred specification, PINK flexible model, GREY context.
BLUE <- "#2A78D6"
ORANGE <- "#EB6834"
GREEN <- "#1BAF7A"
PINK <- "#C2255C"
GREY <- "#8F8D87"

# Task groups keep one color each wherever all four appear together.
TASK_COLORS <- c(
  "Non-Routine Cognitive" = BLUE,
  "Non-Routine Manual" = PINK,
  "Routine Cognitive" = GREEN,
  "Routine Manual" = ORANGE
)

dollar_axis <- function(v) {
  ifelse(is.na(v), "",
    ifelse(v == 0, "$0",
      sprintf("%s$%s", ifelse(v < 0, "-", ""),
              formatC(abs(v), format="d", big.mark=","))))
}

theme_set(
  theme_minimal(base_size=11) +
    theme(
      panel.background=element_rect(fill="#FCFCFB", color=NA),
      plot.background=element_rect(fill="#FCFCFB", color=NA),
      panel.grid.major=element_line(color="#E1E0D9", linewidth=0.35),
      panel.grid.minor=element_blank(),
      plot.title=element_text(face="bold", size=12, color="#0B0B0B", margin=margin(b=4)),
      plot.subtitle=element_text(size=9.5, color="#52514E", margin=margin(b=8)),
      strip.background=element_blank(),
      strip.text=element_text(face="bold", size=10, color="#0B0B0B"),
      axis.line=element_line(color="#C3C2B7", linewidth=0.3),
      axis.ticks=element_blank(),
      axis.text=element_text(size=9, color="#52514E"),
      axis.title=element_text(size=10, color="#0B0B0B"),
      legend.position="bottom",
      legend.title=element_blank(),
      legend.text=element_text(size=9.5, color="#0B0B0B"),
      legend.key=element_blank(),
      plot.margin=margin(10, 14, 8, 10)
    )
)
''')

R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: package ‘ggplot2’ was built under R version 4.5.3 
  

# Introduction

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()
eng=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
intro_yr=2023
d=pd.read_sql("""
select cte.routine_cognitive_share, cte.routine_manual_share
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
where ca.year=%(yr)s
""", eng, params={"yr": intro_yr})
routine=(d["routine_cognitive_share"]+d["routine_manual_share"])*100
routine_mean=routine.mean()
routine_q1=routine.quantile(0.25)
routine_q3=routine.quantile(0.75)

According to Gallup, concern about technology rendering jobs obsolete rose between 2021 and 2023 ([Saad 2023](#ref-Gallup2023)), and Pew reported that Americans expected automation to displace other workers while leaving their own jobs largely intact ([Smith and Anderson 2017](#ref-Pew2017)). Concern of this kind is a belief about the future, and it need not track the work a place contains. The counties analyzed in this study house approximately 282 million people, about 84 percent of the United States population ([U.S. Census Bureau 2023](#ref-census_popest2023)). Across those counties, routine work accounted for an average of 39 percent of task content in 2023, and the middle half fell between 35 and 43 percent. Job loss matters to a household through income, and income matters through what it buys, which varies with local prices. A wage that covers housing in one county falls short in another, so whatever automation means for livelihoods has to show up in what residents can afford. This study asks whether the balance of work in a county is associated with the purchasing power of the people who live there.

The answer bears on decisions made with public money. Poverty and unemployment rates already steer which counties receive distress relief and development investment, and a measurable characteristic of local work that carries information those rates miss would sharpen where that support goes. <a href="#sec-conclusions" class="quarto-xref">Section 6</a> returns to these uses.

Three studies established the framework we build on. Autor, Levy, and Murnane ([2003](#ref-Autor2003)) showed that computers substitute for routine tasks and complement non-routine ones, which made it possible to identify occupations exposed to automation. Applying that idea to local labor markets, Autor and Dorn ([2013](#ref-AutorDorn2013)) documented polarization in routine intensive areas, where employment grew at both the high and low ends of the wage distribution while middle wage employment declined. Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) then generalized the argument into a model in which tasks are the unit of production and skill groups compete to supply them, and later empirical work builds on that model.

Nevertheless, none of this work speaks directly to purchasing power at the county level, for three reasons. Outcomes appear in unadjusted wages and employment counts rather than in what those wages buy locally. The unit of analysis is the commuting zone rather than the county. Estimation is linear, so these designs cannot say whether purchasing power moves with the balance of work in fixed dollar steps or in proportion. These are not flaws in the earlier studies, which were built for other questions. <a href="#sec-background" class="quarto-xref">Section 2</a> takes each one up in turn.

In this study we built a county by year panel covering 2008 to 2023, excluding 2020, from datasets published by federal agencies. Each county year is described by four task groups, routine cognitive, routine manual, non-routine cognitive, and non-routine manual. These follow the task framework of Autor, Levy, and Murnane ([2003](#ref-Autor2003)) and the occupation taxonomy of Autor and Dorn ([2013](#ref-AutorDorn2013)). The outcome is purchasing power, stated in dollars. It divides county median household income by the local price level, which comes from the Bureau of Economic Analysis Regional Price Parities. The panel holds 848 counties and 11,983 county year observations. <a href="#sec-data" class="quarto-xref">Section 3</a> describes the sources and how the panel was built.

We estimated a panel regression with year indicators, controls for poverty, unemployment, and population, and standard errors clustered by county. Clustering accounts for the dependence among repeated observations of the same county. Residual plots, quantile quantile plots, and variance inflation factors tested whether the specification held. Those diagnostics showed that purchasing power moves proportionally rather than in fixed dollar steps, and a log transformation captured most of that structure. Structured deviations remained in the residuals afterward. The regression established the adjusted association, so we then fit random forest and neural network models to test whether relaxing the functional form improved prediction on counties the models had not seen. Every model was scored on the same counties held out by a split grouped at the county level. Complexity increased only where the diagnostics showed a simpler specification falling short, and the added flexibility did not meaningfully improve held out performance. <a href="#sec-analysis" class="quarto-xref">Section 4</a> reports where each model landed.

Our primary finding comes from the log specification, evaluated at the panel mean. A one percentage point shift toward non-routine manual work and away from non-routine cognitive work is associated with \$846 less in purchasing power, with a 95 percent interval of plus or minus \$38.

# Background and related work

This study starts from a change in how economists think about what technology replaces. Autor, Levy, and Murnane ([2003](#ref-Autor2003)) argued that technology replaces tasks rather than whole occupations. Tasks that follow explicit, rule based procedures can be written down as programs, which makes computers close substitutes for the people who perform them. Tasks that call for judgment, adaptation, or response to a situation resist that translation, and technology complements them instead. Sorting tasks along two axes, routine against non-routine and cognitive against manual, yields the four groups used here and turns a broad worry about automation into something a dataset can carry.

Task content measured at the occupational level can also be measured at the place level, because places differ in the work their residents do. Autor and Dorn ([2013](#ref-AutorDorn2013)) showed that local labor markets hold long lasting differences in the work they contain, and that those differences drove employment polarization across United States commuting zones. Routine intensive areas lost middle wage jobs and gained employment at both the low and high ends of the wage distribution. We used their occupation taxonomy when assigning occupations to task groups. Their finding that the balance of work is a persistent feature of a place, rather than a passing feature of its residents, is what makes a panel design the right one here.

Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) gave the approach its formal grounding. In their model tasks are the unit of production, and skill groups compete to supply them. Technological change decides which group holds the comparative advantage in which task, so what a local economy produces depends on the tasks its workers perform. Our county level design rests on that last point.

A separate line of evidence records what workers themselves expect, and it is where this study began. Surveyed concern about automation has risen sharply, even as workers place the displacement on jobs other than their own ([Saad 2023](#ref-Gallup2023); [Smith and Anderson 2017](#ref-Pew2017)). Surveys of this kind measure belief, and belief forms from headlines and personal experience rather than from the occupations a county’s residents hold. Measuring that work directly is the point of what follows.

Three limits stand between these studies and a claim about county purchasing power. The first is that a dollar is not a constant unit. A wage buys different amounts in different places, and prior work reports outcomes in unadjusted wages and employment counts. A high wage metropolitan area and a low wage rural county can look far apart in dollar terms even when those dollars buy about the same goods. <a href="#fig-income-gap" class="quarto-xref">Figure 1</a> shows the size and direction of that distortion. The error runs in both directions, so nominal income reorders the ranking of counties rather than inflating it uniformly.

In [3]:
import os
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine

engine = create_engine(
    os.environ["AUTORACK_URL"],
    pool_pre_ping=True,
    pool_recycle=300
)
gap_county = pd.read_sql("""
    select
        b.county_fips,
        avg(
            b.median_household_income - a.affordability_salary
        ) as income_gap
    from county_baseline b
    join county_affordability a
      on a.county_fips = b.county_fips
     and a.year = b.year
    group by b.county_fips
""", engine)
gap_county["fips"] = (
    gap_county["county_fips"]
    .astype(str)
    .str.zfill(5)
)
gdf = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/"
    "cb_2021_us_county_500k.zip"
)
gdf = gdf.rename(columns={"GEOID": "fips"})
merged_gap = gdf.merge(
    gap_county[["fips", "income_gap"]],
    on="fips",
    how="left"
)
# Continental United States only
merged_gap = merged_gap[
    ~merged_gap["STATEFP"].isin(
        ["02", "15", "60", "66", "69", "72", "78"]
    )
].copy()
# Symmetric color range around zero
gap_bound = float(gap_county["income_gap"].abs().quantile(0.98))
os.makedirs("output", exist_ok=True)
merged_gap[["fips", "income_gap", "geometry"]].to_file(
    "output/income_gap.geojson",
    driver="GeoJSON"
)

In [4]:
%%R -i gap_bound -w 12 -h 7 -u in -r 150
suppressMessages(library(sf))

gap_sf <- st_read("output/income_gap.geojson", quiet=TRUE)

ggplot(gap_sf) +
  geom_sf(aes(fill=income_gap), color="white", linewidth=0.05) +
  scale_fill_gradient2(low="#2166AC", mid="#FFFFFF", high="#B2182B",
                       midpoint=0, limits=c(-gap_bound, gap_bound),
                       oob=scales::squish, na.value="#EEEEEE",
                       name="Income gap ($)",
                       labels=function(v) sprintf("$%+.0fk", v/1000)) +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom",
        legend.key.width=unit(2, "cm"))

The second limit is geography. A commuting zone pools the counties inside it and reports their average, which is the right unit for studying labor market adjustment and the wrong one for studying what a paycheck covers. Housing costs, and with them the gap in <a href="#fig-income-gap" class="quarto-xref">Figure 1</a>, vary sharply between a metropolitan core and the counties around it. Averaging across that boundary removes exactly the variation the question turns on.

The third limit is functional form. Prior estimation is linear, which assumes that a given shift in the balance of work moves purchasing power by the same number of dollars everywhere. If the relationship is proportional instead, that assumption understates the difference in high purchasing power counties and overstates it in low ones, and a linear model cannot distinguish the two cases from the inside.

Together the three mark out what prior work cannot say about purchasing power at the county level, and the data and design in <a href="#sec-data" class="quarto-xref">Section 3</a> address each one.

# Data

## Data collection

The study follows counties over fifteen years, so the data must measure every county on consistent definitions in every year of the panel. Federal statistical programs are the only sources that meet that requirement at this scale. We therefore drew on five datasets published by three federal agencies, each retrieved in June 2026 through an agency application programming interface (API) or bulk file download and loaded into a PostgreSQL database. <a href="#tbl-sources" class="quarto-xref">Table 1</a> reports each source, its record count, and the variables it supplies.

In [5]:
import pandas as pd
from great_tables import GT

sources_df=pd.DataFrame({
    "Source": ["Census SAIPE","BEA Regional Price Parities","Census CBSA delineation",
               "ACS 1 year estimates","BLS LAUS"],
    "Records retrieved": [50283, 884, 387, 15690, 88004],
    "Variables supplied": ["median household income, poverty rate",
                           "county price level relative to the national average",
                           "county to metropolitan area crosswalk",
                           "population, occupational employment by category",
                           "unemployment rate"],
    "Warehouse table": ["`county_baseline`","`cbsa_rpp`","`cbsa`",
                        "`county_baseline`, `county_task_exposure`","`county_baseline`"],
})

(GT(sources_df)
  .fmt_integer(columns="Records retrieved", use_seps=True)
  .fmt_markdown(columns="Warehouse table")
  .cols_align(align="right", columns="Records retrieved")
  .tab_source_note("All sources retrieved June 2026 through agency APIs or bulk file download."))

The five sources arrive at two levels of observation, county by year and metropolitan area. The pipeline transforms each to a common county by year grain, and county Federal Information Processing Standards (FIPS) code and year together serve as the join key across the warehouse. Regional Price Parities required one extra step, because the Bureau of Economic Analysis (BEA) publishes them at the metropolitan area level, so counties were linked to their metropolitan area through the Census Core Based Statistical Area (CBSA) delineation file before joining.

The study period spans 2008 to 2023, excluding 2020. The American Community Survey (ACS) suspended its 1 year estimates that year following COVID related disruptions to data collection. Because those estimates supply the occupational employment counts underlying the task group values, the task groups cannot be constructed for 2020. The Small Area Income and Poverty Estimates (SAIPE) program publishes estimates for approximately 3,143 counties per year, giving 47,140 county year observations across the fifteen study years.[1]

## Measuring task groups

Following the task framework introduced in <a href="#sec-background" class="quarto-xref">Section 2</a>, we assigned each of the broad occupational categories published in ACS 1 year estimates to one of four task groups, routine cognitive, routine manual, non-routine cognitive, and non-routine manual. The Census categories carry no such labels, so the assignment follows the classification logic of Autor and Dorn ([2013](#ref-AutorDorn2013)), with clerical and sales work as routine cognitive, production and construction work as routine manual, managerial, professional, and technical work as non-routine cognitive, and service work as non-routine manual. For each county and year, the task group totals are the employment sums across the categories assigned to each group, shown in <a href="#eq-totals" class="quarto-xref">Equation 1</a>.

<span id="eq-totals">$$
T_{g,c,t} = \sum_{o \in g} E_{o,c,t}
 \qquad(1)$$</span>

Here $E_{o,c,t}$ is employment in occupational category $o$ in county $c$ during year $t$, and the sum runs over the categories assigned to group $g$. No task rating weights enter the calculation. Each group total is a simple employment count over the categories assigned to it, so a county’s four values describe what its workforce does rather than a rated intensity of any task. Dividing the four totals by their sum, as in <a href="#eq-group" class="quarto-xref">Equation 2</a>, describes each county year by four values on a zero to one scale that sum to one. Together these four values are the county’s task composition.

<span id="eq-group">$$
G_{g,c,t} = \frac{T_{g,c,t}}{\sum_{g'} T_{g',c,t}}
 \qquad(2)$$</span>

Normalizing by the four group total turns employment counts into proportions, which puts counties of very different sizes on a common scale and makes the balance of work the quantity of interest rather than its volume. <a href="#sec-analysis" class="quarto-xref">Section 4</a> reports what this means for estimation.

Retaining four separate groups, rather than collapsing them into a single routine intensity index, preserves a distinction that a composite measure erases. Under one index, a county shifting away from clerical work and a county shifting away from assembly work look identical.

One property of the source categories matters later. The Census Bureau revised its occupational classification between 2009 and 2010, so category boundaries differ slightly across that break even though the four group assignment is unchanged. <a href="#sec-analysis" class="quarto-xref">Section 4</a> reports the consequences for measured variation and the robustness checks that address them.

## Measuring purchasing power

The outcome variable is purchasing power, measured in dollars and defined in <a href="#eq-afford" class="quarto-xref">Equation 3</a> as county median household income, from the Census SAIPE program, divided by the local Regional Price Parity. BEA publishes parities as an index with the national average set to 100, so dividing by 100 converts the index into a price multiplier.

<span id="eq-afford">$$
A_{c,t} = \frac{\text{Median household income}_{c,t}}{\text{RPP}_{c,t} / 100}
 \qquad(3)$$</span>

A county with a median household income of \$60,000 and a price parity of 120 has a purchasing power of \$50,000, meaning local income buys what \$50,000 would buy at national average prices. Stated this way, income is comparable across counties, since a given figure purchases roughly the same bundle wherever it appears.

Regional Price Parities correct for price differences between places but not between years, so purchasing power is not deflated to a constant base year and the series is in nominal dollars. Year indicators in the regression absorb whatever is common to all counties in a given year, national price movement included, and they do so exactly, with one free parameter per year rather than a relationship the model has to learn. Deflating by a national index would divide every county’s outcome in a year by the same factor, which the year indicators already remove, so in the log specification the task group coefficients are identical with or without deflation. What deflation would change is the year effects. Without it they carry price level drift alongside any other national change, and we report them as nuisance parameters rather than as evidence of rising purchasing power.

Counties outside any metropolitan area have no published parity of their own and take their state parity instead. Within the analytical panel, 45.5 percent of county year rows use a metropolitan parity and 54.5 percent use the state fallback. This is a substantive limitation, since a single state level price index is applied to counties whose actual price levels differ, and a majority of panel rows carry it. The counties involved are above the 65,000 population threshold but predominantly outside metropolitan statistical areas, where BEA publishes no local parity. The analysis does not separate fallback rows from metropolitan ones, so the estimates below are averages across both.

## Control variables

Three baseline county conditions enter the models as controls. Unemployment rate comes from the Bureau of Labor Statistics Local Area Unemployment Statistics (BLS LAUS) program and poverty rate from the Census SAIPE program, both measured in percentage points. Total population comes from ACS 1 year estimates and enters the models in logs. Unemployment and poverty capture labor market slack and material hardship that the task groups would otherwise absorb. Population accounts for scale differences across counties, which range from the ACS publication threshold of 65,000 to the roughly 10 million residents of Los Angeles County.

## Storage and organization

Data are stored in PostgreSQL across two databases that serve different purposes. The first is a data lake of 41 tables holding raw retrievals and intermediate results, deliberately left unnormalized. Its purpose is traceability. Any figure in this report can be traced back to the records it came from, and any processing step can be rerun without returning to an agency API, including retrievals for analysis branches later scoped out.

The second is the analytical warehouse of eleven tables, whose analytical core is in third normal form. All cleaning occurs upstream, so the warehouse is effectively read only from the perspective of the analysis. Cleaning between the lake and the warehouse standardized county identifiers to a single type, restricted records to the study period, removed jurisdictions outside the panel scope, and enforced foreign key constraints so that every fact row joins to a known county. <a href="#fig-erd" class="quarto-xref">Figure 2</a> traces the full pipeline from source to output; its center expands the warehouse layer to show the eight tables the analysis reads. A population staging table, a compatibility view, and an employment counts table retained from an earlier analysis branch are omitted.

[1] Two jurisdictions present in SAIPE for all fifteen years, the District of Columbia (FIPS 11001) and Kalawao County, Hawaii (FIPS 15005), are absent from the county reference file and are therefore excluded from the panel. The District of Columbia exceeds the population threshold described below and would otherwise have been retained. Connecticut requires a related note. The state replaced its eight counties with nine planning regions beginning in 2022, and only the legacy counties were ingested, so Connecticut counties exit the panel after 2021 rather than continuing under successor codes.

In [6]:
import graphviz

def entity(name, rows):
    body = f'<TR><TD COLSPAN="3" BGCOLOR="#2A78D6"><FONT COLOR="white" POINT-SIZE="11"><B>{name}</B></FONT></TD></TR>'
    for typ, col, marker in rows:
        m = f'<FONT COLOR="#8F8D87" POINT-SIZE="8">{marker}</FONT>' if marker else ""
        body += f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">{typ}</FONT></TD><TD ALIGN="LEFT"><FONT POINT-SIZE="10">{col}</FONT></TD><TD ALIGN="LEFT">{m}</TD></TR>'
    return f'<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="2">{body}</TABLE>>'

dot = f'''
digraph architecture {{
  rankdir=TB
  bgcolor="#FCFCFB"
  fontname="Helvetica"
  nodesep=0.22
  ranksep=0.3
  node [fontname="Helvetica", shape=box, style="rounded", color="#C3C2B7"]
  edge [fontname="Helvetica", color="#8F8D87", arrowsize=0.6]

  subgraph cluster_source {{
    label="DATA SOURCES"
    labeljust="l"
    fontsize=11
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    node [style="filled,rounded", fillcolor="white", margin="0.1,0.05", fontsize=10]
    src_census [label="Census Bureau\\nSAIPE · ACS 1yr · CBSA delineation"]
    src_bea [label="BEA\\nRegional Price Parities"]
    src_bls [label="BLS\\nLAUS"]
  }}

  subgraph cluster_lake {{
    label="DATA LAKE"
    labeljust="l"
    fontsize=11
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    node [style="filled,rounded", fillcolor="white", margin="0.1,0.05", fontsize=10]
    lake [label="Raw retrievals and intermediate results\\n41 tables, unnormalized"]
  }}

  subgraph cluster_warehouse {{
    label="DATA WAREHOUSE — THIRD NORMAL FORM"
    labeljust="l"
    fontsize=11
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    node [shape=plain]

    state [label={entity("state", [("varchar","state_code","PK"),("varchar","state_name","")])}]
    county [label={entity("county", [("bigint","county_fips","PK"),("varchar","county_name",""),("varchar","state_code","FK")])}]
    county_baseline [label={entity("county_baseline", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","population",""),("numeric","median_household_income",""),("numeric","poverty_rate",""),("numeric","unemployment_rate","")])}]
    county_task_exposure [label={entity("county_task_exposure", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","routine_cognitive",""),("numeric","routine_manual",""),("numeric","non_routine_cognitive",""),("numeric","non_routine_manual",""),("numeric","routine_cognitive_share","generated"),("numeric","routine_manual_share","generated"),("numeric","non_routine_cognitive_share","generated"),("numeric","non_routine_manual_share","generated")])}]
    county_affordability [label={entity("county_affordability", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","affordability_salary","")])}]
    county_cbsa_code [label={entity("county_cbsa_code", [("bigint","county_fips","PK,FK"),("varchar","cbsa_code","PK,FK")])}]
    cbsa [label={entity("cbsa", [("varchar","cbsa_code","PK"),("varchar","cbsa_name","")])}]
    cbsa_rpp [label={entity("cbsa_rpp", [("varchar","cbsa_code","PK,FK"),("int","year","PK"),("numeric","rpp_value","")])}]

    {{rank=same; county_baseline; county_task_exposure; county_affordability}}

    state -> county
    county -> county_baseline
    county -> county_task_exposure
    county -> county_affordability
    county -> county_cbsa_code
    county_cbsa_code -> cbsa
    cbsa -> cbsa_rpp
  }}

  subgraph cluster_output {{
    label="ANALYSIS OUTPUTS"
    labeljust="l"
    fontsize=11
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    node [style="filled,rounded", fillcolor="white", margin="0.1,0.05", fontsize=10]
    analysis_table [label="Analysis ready county-year table\\n(joined on county_fips and year)"]
    models [label="Statistical and ML models"]
    figures [label="Figures and tables"]
    {{rank=same; models; figures}}
    analysis_table -> models
    analysis_table -> figures
  }}

  src_census -> lake
  src_bea -> lake
  src_bls -> lake
  lake -> state
  county_baseline -> analysis_table [label="  joined with the other fact tables  ", fontsize=8, fontcolor="#8F8D87"]
  county_task_exposure -> analysis_table [style=invis]
  county_affordability -> analysis_table [style=invis]
}}
'''
graphviz.Source(dot)

The schema centers on a `county` table keyed by FIPS code, with `state` as a reference table. Three fact tables join to `county` on FIPS code and year. `county_baseline` holds population, median household income, poverty rate, and unemployment rate, `county_task_exposure` holds the four task group totals from <a href="#eq-totals" class="quarto-xref">Equation 1</a>, and `county_affordability` holds the outcome variable.

Two table names are historical. `county_task_exposure` dates from an earlier measure built on Occupational Information Network (O\*NET) task ratings that was not carried forward, and the values it holds are the employment totals of <a href="#eq-totals" class="quarto-xref">Equation 1</a>. `county_affordability` and its `affordability_salary` column predate the purchasing power terminology used throughout this report, and the quantity stored is the one defined in <a href="#eq-afford" class="quarto-xref">Equation 3</a>. The normalized values from <a href="#eq-group" class="quarto-xref">Equation 2</a> are not computed in analysis code but stored as generated columns, which the database derives from the four totals on every write. The equation is enforced in the schema itself, so no analysis can read a stale or inconsistently normalized value.

Price parities require a separate path. Metropolitan membership is kept out of the `county` table and in the `county_cbsa_code` junction table, so counties outside any metropolitan area carry no null CBSA fields, and the crosswalk stands alone as its own fact. From there `cbsa` connects to `cbsa_rpp`. Storing parities at the CBSA level reflects the level at which BEA measures them and avoids repeating a single value across every county in a metropolitan area.

## The analytical dataset

The 47,140 county year observations described above are the records the source data support, not the records the models estimate on. Two filters reduce this frame, and <a href="#fig-funnel" class="quarto-xref">Figure 3</a> traces them.

In [7]:
funnel_df = pd.DataFrame({
    "stage": ["Source frame", "Task group coverage", "Estimation sample"],
    "x": [1, 2, 3],
    "value": [47140, 12087, 11983],
})
excl_df = pd.DataFrame({
    "x": [1.5, 2.5],
    "removed": [47140 - 12087, 12087 - 11983],
    "reason": ["below 65,000 residents", "missing unemployment rate"],
})

In [8]:
%%R -i funnel_df -i excl_df -w 8 -h 3 -u in -r 150
TEAL <- "#1E8E99"; ORANGE <- "#EB6834"; GREY <- "#8F8D87"

funnel_df$color <- c(GREY, GREY, TEAL)
funnel_df$fontface <- c("plain", "plain", "bold")
funnel_df$label <- paste0(funnel_df$stage, "\n", format(funnel_df$value, big.mark=","))
excl_df$label <- paste0("−", format(excl_df$removed, big.mark=","), "\n", excl_df$reason)

ggplot(funnel_df, aes(x=x, y=0)) +
  geom_line(linewidth=0.9, color=GREY) +
  geom_point(aes(color=color), size=6) +
  scale_color_identity(guide="none") +
  geom_text(aes(label=label, fontface=fontface), vjust=-1.6, size=3.4, lineheight=0.95) +
  geom_text(data=excl_df, aes(x=x, y=0, label=label), vjust=2.5, size=3.15, color=ORANGE, lineheight=0.95) +
  scale_x_continuous(limits=c(0.6, 3.4)) +
  scale_y_continuous(limits=c(-1.6, 1.9)) +
  theme_void()

Task group coverage is the binding constraint. The occupational employment counts that build the four task groups come from ACS 1 year estimates, which the Census Bureau publishes only for areas with populations above 65,000. Counties below that threshold have no occupational estimates in any year, so no task groups can be constructed for them. This leaves 12,087 county year observations across 848 counties, roughly a quarter of the 47,140.

Requiring non missing controls removes a further 104 observations. Every one of the 104 is missing an unemployment rate, giving BLS LAUS a coverage rate of 99.1 percent across the retained counties. Population, median household income, and poverty rate are complete. The estimation sample is therefore 11,983 county year observations, and all models reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a> and <a href="#sec-results" class="quarto-xref">Section 5</a> fit on these same rows.

The 848 counties are also the unit the machine learning models split on. Because a county appears in the panel up to fifteen times, splitting rows at random would place the same county in both training and test data, so the split is grouped by county and every observation of a given county falls entirely on one side. This makes 848, rather than 11,983, the effective sample size for held out evaluation. <a href="#sec-analysis" class="quarto-xref">Section 4</a> takes up the reasoning behind the grouped split.

The population threshold limits what the results can describe. The 848 retained counties hold about 84 percent of the United States population but a minority of its counties, and they are more metropolitan, more populous, and more economically diverse than those dropped. Rural counties are underrepresented as a direct consequence. The restriction covers the regression coefficients and the machine learning results alike, so the partial dependence curves and importance rankings in <a href="#sec-results" class="quarto-xref">Section 5</a> describe how the models behave across counties above 65,000 residents rather than across counties in general. <a href="#sec-conclusions" class="quarto-xref">Section 6</a> returns to this limitation and to one approach for addressing it in future work.

# Analysis

In [9]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
df=pd.read_sql("""
select ca.county_fips, ca.year, ca.affordability_salary,
    cte.routine_cognitive_share, cte.routine_manual_share,
    cte.non_routine_cognitive_share, cte.non_routine_manual_share,
    cb.poverty_rate, cb.unemployment_rate, cb.population
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
""", engine)
df["log_population"]=np.log(df["population"])

In [10]:
# level model: OLS with year indicators, standard errors clustered by county
reg_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

reg=df.dropna(subset=reg_cols+["affordability_salary"]).copy()

X_panel=pd.concat([reg[reg_cols],
                   pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_panel=sm.add_constant(X_panel)

model=sm.OLS(reg["affordability_salary"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

reg["fitted"]=model.fittedvalues
reg["resid"]=model.resid

# log respecification, same design matrix
reg["actual_log"]=np.log(reg["affordability_salary"])
model_log=sm.OLS(reg["actual_log"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})
reg["resid_log"]=model_log.resid
reg["fitted_log"]=model_log.fittedvalues

## The unadjusted relationship

The ladder starts with the relationship itself, before any controls. <a href="#fig-univariate" class="quarto-xref">Figure 4</a> plots purchasing power against each of the four task groups across the panel. The two manual groups slope down most steeply, non-routine cognitive slopes up, and routine cognitive is the flattest of the four. None of the four is tight enough to carry a claim on its own, which is the point of the models that follow, but the ordering visible here is the ordering the adjusted estimates preserve.

In [11]:
uni_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
uni_pts=df.melt(id_vars="affordability_salary", value_vars=list(uni_labels.keys()),
                var_name="group", value_name="share").dropna()
uni_pts["group"]=uni_pts["group"].map(uni_labels)
uni_pts["share"]=uni_pts["share"]*100  # express as percent of county employment

uni_pts["bin"]=uni_pts.groupby("group")["share"].transform(
    lambda s: pd.qcut(s, 20, labels=False, duplicates="drop"))
uni_bins=(uni_pts.groupby(["group","bin"])[["share","affordability_salary"]]
          .mean().reset_index())
uni_order=list(uni_labels.values())

In [12]:
%%R -i uni_pts -i uni_bins -i uni_order -w 10 -h 6 -u in -r 150
uni_pts$group <- factor(uni_pts$group, levels=unlist(uni_order))
uni_bins$group <- factor(uni_bins$group, levels=unlist(uni_order))

ggplot(uni_pts, aes(x=share, y=affordability_salary)) +
  geom_point(alpha=0.03, size=0.25, color="grey60") +
  geom_line(data=uni_bins, color="#0072B2", linewidth=0.8) +
  geom_point(data=uni_bins, color="#0072B2", size=1.8) +
  facet_wrap(~group, scales="free_x", ncol=2) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(v, format="d", big.mark=","))) +
  labs(x="Percent of County Employment", y="Purchasing Power")

## Panel regression

The four task group values defined in <a href="#eq-group" class="quarto-xref">Equation 2</a> sum to one, so all four cannot enter a regression together. One serves as the reference group, and we use non-routine cognitive. Each remaining coefficient is the change in purchasing power associated with a shift out of non-routine cognitive work and into that group, holding the controls fixed, and each is therefore a comparison against the reference rather than a standalone quantity. The group values run on a zero to one scale, so the per percentage point figures reported later divide the estimated coefficients by 100.

Year indicators capture conditions common to all counties within a year. These include the 2008 recession, the recovery through the 2010s, and the price movement that purchasing power does not itself account for. Because purchasing power is not deflated to a constant base year, the year effects carry price level drift alongside any other national change, and we treat them as nuisance parameters rather than as evidence of rising purchasing power. The task group coefficients compare counties within a year and are unaffected by this.

Standard errors are clustered by county, because repeated observations of the same county across fifteen years are not independent draws. A Durbin Watson statistic of 0.496 is consistent with that dependence. <a href="#sec-results" class="quarto-xref">Section 5</a> reports the estimated coefficients.

## Collinearity and the reference group

The unnormalized employment totals of <a href="#eq-totals" class="quarto-xref">Equation 1</a> cannot support a regression. A variance inflation factor measures how much a coefficient is destabilized by its correlation with the other inputs, and a value above 10 is the conventional threshold for concern. The four totals carry factors between 13 and 27, because all four are employment counts that scale with county size, so they move together and carry little independent information. The specification estimated fixes this twice over, as <a href="#fig-vif" class="quarto-xref">Figure 5</a> shows. Normalizing by the four group total, as <a href="#sec-data" class="quarto-xref">Section 3</a> describes, removes the shared scale, and dropping non-routine cognitive as the reference group removes the constraint that the four proportions sum to one. Every factor on the estimated specification sits between 1.2 and 1.6.

In [13]:
raw=pd.read_sql("""
    select routine_cognitive, routine_manual, non_routine_cognitive, non_routine_manual
    from county_task_exposure
    """, engine)

raw_cols=["routine_cognitive","routine_manual","non_routine_cognitive","non_routine_manual"]
X_raw=sm.add_constant(raw[raw_cols])
vif_raw=pd.Series([variance_inflation_factor(X_raw.values, i) for i in range(1, X_raw.shape[1])],
                  index=raw_cols)

X_vif_level=sm.add_constant(reg[reg_cols])
vif_share=pd.Series([variance_inflation_factor(X_vif_level.values, i) for i in range(1, X_vif_level.shape[1])],
                    index=reg_cols)

group_display=["Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual"]

vif_dot_df=pd.concat([
    pd.DataFrame({"group":group_display, "vif":vif_raw.values,
                  "spec":"Unnormalized Employment Totals"}),
    pd.DataFrame({"group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
                  "vif":vif_share[["routine_cognitive_share","routine_manual_share",
                                   "non_routine_manual_share"]].values,
                  "spec":"Group Proportions, as Estimated"}),
], ignore_index=True)

In [14]:
%%R -i vif_dot_df -w 8 -h 3.5 -u in -r 150
vif_dot_df$group <- factor(vif_dot_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual",
                 "Non-Routine Cognitive","Non-Routine Manual")))
vif_dot_df$spec <- factor(vif_dot_df$spec,
    levels=c("Unnormalized Employment Totals","Group Proportions, as Estimated"))

ggplot(vif_dot_df, aes(x=vif, y=group, color=spec)) +
  geom_vline(xintercept=10, linetype="dashed", linewidth=0.4) +
  geom_line(aes(group=group), color="grey80", linewidth=0.6) +
  geom_point(size=3) +
  geom_text(aes(label=sprintf("%.1f", vif)), vjust=-1.0, size=3.2, show.legend=FALSE) +
  scale_x_log10(limits=c(1, 40)) +
  scale_color_manual(values=c("Unnormalized Employment Totals"="#D55E00",
                              "Group Proportions, as Estimated"="#009E73")) +
  labs(x="Variance Inflation Factor (Log Scale)", y=NULL, color=NULL) +
  theme(legend.position="bottom")

This also explains a problem visible earlier in the project. Before normalization the coefficients were unstable and their standard errors were large relative to their magnitudes, both standard symptoms of multicollinearity.

## Validation before modeling

Before estimating, we check that the data support the chosen methods. <a href="#fig-dists" class="quarto-xref">Figure 6</a> shows the distributions of purchasing power and the four task groups across the task group panel, looking for the skewness and outliers that would shape the specification. The variance inflation factors above already establish that no two groups carry the same information once the proportions are used.

In [15]:
dist_label_map={
    "affordability_salary":"Purchasing Power ($)",
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
dists_df=df.melt(value_vars=list(dist_label_map.keys()),
                 var_name="variable", value_name="value")
dists_df["variable"]=dists_df["variable"].map(dist_label_map)

In [16]:
%%R -i dists_df -w 10 -h 5.5 -u in -r 150
make_hist <- function(data, var_name, fill_color) {
  ggplot(subset(data, variable==var_name), aes(x=value)) +
    geom_histogram(bins=50, fill=fill_color) +
    labs(title=var_name, x=NULL, y="Count")
}

p_pp  <- make_hist(dists_df, "Purchasing Power ($)", BLUE) + scale_x_continuous(labels=dollar_axis)
p_rc  <- make_hist(dists_df, "Routine Cognitive", GREEN) + scale_x_continuous(labels=percent)
p_rm  <- make_hist(dists_df, "Routine Manual", ORANGE) + scale_x_continuous(labels=percent)
p_nrc <- make_hist(dists_df, "Non-Routine Cognitive", BLUE) + scale_x_continuous(labels=percent)
p_nrm <- make_hist(dists_df, "Non-Routine Manual", PINK) + scale_x_continuous(labels=percent)

right_grid <- (p_rc | p_rm) / (p_nrc | p_nrm)
(p_pp | right_grid) + plot_layout(widths=c(1, 1.6))

Purchasing power is right skewed, with a skewness of 1.17. The center of the distribution sits near \$57,000, and a long tail of high purchasing power counties extends above it. That skew is why the diagnostics below concentrate on the extremes.

## Specification diagnostics

A linear model assumes a constant rate of association across the data range and errors that are symmetric and evenly spread around the fit. We test those assumptions with three diagnostics, the residuals plotted against fitted values, a QQ plot of the residuals, and the variance inflation factors already reported. The raw scale model shows clear systematic curvature in the residuals and a substantial upper tail departure in the QQ plot, indicating that the linear specification is poorly suited to the untransformed outcome; <a href="#fig-diagnostics" class="quarto-xref">Figure 8</a> shows both, alongside the same diagnostics after the log respecification below. The model underestimates both the lowest and the highest purchasing power counties while fitting the middle well, and the upper tail departure reflects a residual skew of 1.01.

The level model still supplies the interpretable dollar estimates we report, and its own diagnostics show that the data hold structure a straight line in dollars cannot represent. These diagnostics are what drive the rest of the section. Curvature in the residual smoother and widening spread with fitted values are evidence that the fixed slope specification is wrong in a specific way, not merely imperfect, and each subsequent model is an attempt to relax the assumption the diagnostics identify.

## Log respecification

The level model’s failure suggests that a constant dollar association is a poor description of the relationship. Logging the outcome tests the alternative, that the associations are proportional, so the same shift in an input moves a high purchasing power county by more dollars than a low one. If that description fits better, the transformation should resolve the residual pattern the level model leaves behind. <a href="#fig-log-transform" class="quarto-xref">Figure 7</a> shows the outcome before and after the transformation.

In [17]:
level_label=f"Level, Skew {reg['affordability_salary'].skew():.2f}"
log_label=f"Logged, Skew {reg['actual_log'].skew():.2f}"
logt_df=pd.concat([
    pd.DataFrame({"value":reg["affordability_salary"], "dist":level_label}),
    pd.DataFrame({"value":reg["actual_log"], "dist":log_label}),
], ignore_index=True)
logt_order=[level_label, log_label]

In [18]:
%%R -i logt_df -i logt_order -w 11 -h 3.5 -u in -r 150
logt_df$dist <- factor(logt_df$dist, levels=unlist(logt_order))

ggplot(logt_df, aes(x=value, fill=dist)) +
  geom_histogram(bins=50, show.legend=FALSE) +
  facet_wrap(~dist, scales="free") +
  scale_fill_manual(values=setNames(c("#D55E00","#009E73"), unlist(logt_order))) +
  labs(x=NULL, y="Count")

Refitting the same specification on log purchasing power resolves most of the failure. <a href="#fig-diagnostics" class="quarto-xref">Figure 8</a> compares the residual and QQ diagnostics for both specifications directly. After logging purchasing power, the residual pattern becomes substantially flatter and the QQ points track the reference line more closely, though some tail deviation remains; the residual skew falls from 1.01 to 0.09. The log linear fit is the baseline against which we evaluate the flexible models below.

In [19]:
resid_df=reg[["fitted","resid"]].copy()
resid_log_df=reg[["fitted_log","resid_log"]].copy()

In [20]:
%%R -i resid_df -i resid_log_df -w 9 -h 7 -u in -r 150
resid_df$std_resid <- as.numeric(scale(resid_df$resid))
resid_log_df$std_resid <- as.numeric(scale(resid_log_df$resid_log))

p1 <- ggplot(resid_df, aes(x=fitted, y=resid)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8EEF4", high=BLUE, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=ORANGE, linewidth=0.9) +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis) +
  labs(x=NULL, y="Residual, Raw ($)", title="Residuals vs. Fitted")

p2 <- ggplot(resid_df, aes(sample=std_resid)) +
  stat_qq(color=BLUE, alpha=0.35, size=0.8) +
  stat_qq_line(color=ORANGE, linewidth=0.8) +
  labs(x=NULL, y="Standardized Residual", title="Normal Q-Q")

p3 <- ggplot(resid_log_df, aes(x=fitted_log, y=resid_log)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8EEF4", high=BLUE, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=ORANGE, linewidth=0.9) +
  labs(x="Fitted Purchasing Power", y="Residual, Log")

p4 <- ggplot(resid_log_df, aes(sample=std_resid)) +
  stat_qq(color=BLUE, alpha=0.35, size=0.8) +
  stat_qq_line(color=ORANGE, linewidth=0.8) +
  labs(x="Theoretical Quantiles", y="Standardized Residual")

(p1 | p2) / (p3 | p4)

<a href="#fig-binned-scale" class="quarto-xref">Figure 9</a> puts the same comparison in the units of the outcome. Fitted values are cut into twenty equal count bins, and each point compares a bin’s mean prediction with its mean actual value, so points on the line indicate no systematic bias in that range. On the level scale the model tracks purchasing power closely through the middle of the distribution, where most counties sit, while both tail bins come in roughly \$8,000 to \$9,000 above their predictions. On the log scale the points track the line throughout.

In [21]:
reg.loc[:, "bin"]=pd.qcut(reg["fitted"], 20, labels=False)
binned=reg.groupby("bin")[["fitted","affordability_salary"]].mean()
reg.loc[:, "bin_log"]=pd.qcut(reg["fitted_log"], 20, labels=False)
binned_log=reg.groupby("bin_log")[["fitted_log","actual_log"]].mean()

pts_df=pd.concat([
    pd.DataFrame({"fitted":reg["fitted"], "actual":reg["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":reg["fitted_log"], "actual":reg["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)
bins_df=pd.concat([
    pd.DataFrame({"fitted":binned["fitted"], "actual":binned["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":binned_log["fitted_log"], "actual":binned_log["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)

In [22]:
%%R -i pts_df -i bins_df -w 13 -h 5.5 -u in -r 150
pts_df$target <- factor(pts_df$target, levels=c("Level Target","Log Target"))
bins_df$target <- factor(bins_df$target, levels=c("Level Target","Log Target"))

ggplot(pts_df, aes(x=fitted, y=actual)) +
  geom_point(alpha=0.05, size=0.3, color="grey50") +
  geom_abline(slope=1, intercept=0, linewidth=0.4) +
  geom_point(data=bins_df, aes(color=target), size=2.4, show.legend=FALSE) +
  facet_wrap(~target, scales="free") +
  scale_color_manual(values=c("Level Target"="#D55E00","Log Target"="#009E73")) +
  labs(x="Fitted Purchasing Power", y="Actual Purchasing Power")

The smoothed residual line does not lie perfectly flat even after the transformation. What remains could be nonlinearity, interactions among the inputs, or a variable the model does not contain, and the residuals alone cannot separate the three. That is exactly the situation a flexible model is built for. A random forest searches all three at once without being told in advance which to look for, so it tests the possibilities the diagnostics raise but cannot settle. It also fixes the standard the forest must meet, which is finding signal in held out counties that the log linear fit does not already capture.

## Variance decomposition

Purchasing power and the task groups vary both across counties and within a county over time, and the two carry different implications. <a href="#fig-between-within" class="quarto-xref">Figure 10</a> splits each group’s variation into the two components. The manual groups vary almost entirely between counties, which is what makes the purchasing power differences they carry durable rather than temporary. Routine cognitive is the only group with substantial within county movement.

In [23]:
df10=df[df["year"]>=2010].copy()
input_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

within_vals=[]
for col in input_cols:
    yd=df10[col]-df10.groupby("year")[col].transform("mean")
    within=(yd-yd.groupby(df10["county_fips"]).transform("mean")).var()
    within_vals.append(within/yd.var()*100)

bw_labels=["Routine Cognitive","Routine Manual","Non-Routine Manual"]
bw_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "value":[100-v for v in within_vals],
                  "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "value":within_vals,
                  "component":"Within Counties Over Time"}),
], ignore_index=True)
bw_text_df=pd.DataFrame({"group":bw_labels,
                         "x":[100-v/2 for v in within_vals],
                         "label":[f"{v:.0f}%" for v in within_vals]})

In [24]:
%%R -i bw_df -i bw_text_df -w 9 -h 3.5 -u in -r 150
bw_df$group <- factor(bw_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual","Non-Routine Manual")))
bw_df$component <- factor(bw_df$component,
    levels=c("Within Counties Over Time","Between Counties"))
bw_text_df$group <- factor(bw_text_df$group, levels=levels(bw_df$group))

ggplot(bw_df, aes(x=value, y=group, fill=component)) +
  geom_col(width=0.7) +
  geom_text(data=bw_text_df, aes(x=x, y=group, label=label),
            inherit.aes=FALSE, color="white", size=3.4) +
  scale_fill_manual(values=c("Between Counties"="grey75",
                             "Within Counties Over Time"="#0072B2"),
                    breaks=c("Between Counties","Within Counties Over Time")) +
  labs(x="Share of Variance (Percent)", y=NULL, fill=NULL) +
  theme(legend.position="bottom")

The Census occupation coding change between 2009 and 2010, described in <a href="#sec-data" class="quarto-xref">Section 3</a>, inflates apparent within county variation for the routine cognitive group. That inflation comes from how the source data were coded rather than from any property of the counties, so the figure is computed on the panel restricted to 2010 onward with each year’s cross county mean removed. We also refit the main estimates on pre pandemic (2010 to 2019) and post pandemic (2021 to 2023) windows, and <a href="#sec-results" class="quarto-xref">Section 5</a> reports coefficient stability across them.

## Predictive modeling

Model complexity follows a ladder, from linear models through tree ensembles to neural networks. The rule is to start with the simplest model that could plausibly work, move up a rung only while held out error improves, and stop when training and validation performance agree. The level model’s failed diagnostics justified the first step, the log respecification. The structured residuals that remain justify testing the next rung, a random forest, which represents interactions and curvature without requiring either to be specified in advance. A small neural network sits one rung above the forest and bounds the search, since if the forest finds no additional signal, the network confirms whether that ceiling is real. Every rung is scored on the same held out counties from the grouped split, so a gain at any rung is attributable to the model rather than to the rows it saw. Under this rule, finding no improvement is itself a result, because it establishes that the association is close to proportional and that the log linear model is the right stopping point.

We reuse the county year panel assembled above, extended with state identifiers for the comparison below.

In [25]:
# state_code isn't in the panel regression's df, add it here
state_lookup=pd.read_sql(
    "select c.county_fips, c.state_code, s.state_name "
    "from county c join state s on c.state_code=s.state_code", engine
)
state_name_map=state_lookup.drop_duplicates("state_code").set_index("state_code")["state_name"]
df_ml=df.merge(state_lookup, on="county_fips", how="left")

### Feature set and reference groups

In [26]:
feature_cols=[
    "routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share",
    "poverty_rate",
    "unemployment_rate",
    "year",
    "state_code",
]

model_df=df_ml.dropna(subset=feature_cols).copy()

The database computes the task groups as each group’s count divided by the four group sum, so they already satisfy the constraint used in the panel regression and need no further normalization here. As before, one group must be dropped to avoid a deterministic column, and we retain `non_routine_cognitive_share` as the omitted reference for consistency with the panel regression.

### Year and state indicators

Treating year as a factor rather than as a numeric variable gives each year its own coefficient, with no assumption about ordering or spacing between them. One state serves as a reference in the same way. Every state with fewer than five distinct counties in the panel is relabeled as OTHER, since those states cannot support a stable coefficient of their own.

In [27]:
model_df["year"]=model_df["year"].astype("category")
model_df["state_code"]=model_df["state_code"].astype("category")

REFERENCE_YEAR=model_df["year"].cat.categories.min()  # earliest year as reference

# pool states with too few counties to support a stable, independent coefficient
county_counts=model_df.groupby("state_code", observed=True)["county_fips"].nunique()
THIN_STATE_THRESHOLD=5
thin_states=county_counts[county_counts<THIN_STATE_THRESHOLD].index.tolist()

model_df["state_code_grouped"]=model_df["state_code"].astype(str)
model_df["state_code_grouped"]=model_df["state_code_grouped"].where(
    ~model_df["state_code_grouped"].isin(thin_states), "OTHER"
)
model_df["state_code_grouped"]=model_df["state_code_grouped"].astype("category")

state_affordability=model_df.groupby("state_code_grouped", observed=True)["affordability_salary"].mean().sort_values()

median_state=state_affordability.index[len(state_affordability)//2]
REFERENCE_STATE=median_state

year_dummies=pd.get_dummies(model_df["year"], prefix="year")
year_dummies=year_dummies.drop(columns=[f"year_{REFERENCE_YEAR}"])

state_dummies=pd.get_dummies(model_df["state_code_grouped"], prefix="state")
state_dummies=state_dummies.drop(columns=[f"state_{REFERENCE_STATE}"])

model_df=pd.concat([model_df, year_dummies, state_dummies], axis=1)

year_cols_model=list(year_dummies.columns)
state_cols_model=list(state_dummies.columns)

The reference state is the one whose mean purchasing power falls at the median of the state means, which is Texas. Texas is not a cost of living outlier in either direction, and it holds considerable within state diversity, with major metropolitan areas alongside a large number of rural counties, so reading other states as different from Texas gives an interpretable baseline. <a href="#fig-state-afford" class="quarto-xref">Figure 11</a> shows where it sits.

In [28]:
state_df=state_affordability.reset_index()
state_df.columns=["state","mean_pp"]
state_df["state_label"]=state_df["state"].map(state_name_map)
# the pooled thin-state bucket is not a state; excluded from the ranking display below
state_df=state_df[state_df["state"]!="OTHER"].copy()
state_median=float(state_affordability.median())
state_df["comparison"]=state_df["mean_pp"].ge(state_median).map({True:"Above Average", False:"Below Average"})

In [29]:
%%R -i state_df -i state_median -w 8 -h 10 -u in -r 150
state_df$state_label <- factor(state_df$state_label, levels=state_df$state_label)
median_label <- sprintf("Median of state means: $%.1fk", state_median/1000)

ggplot(state_df, aes(x=mean_pp, y=state_label, color=comparison)) +
  geom_vline(xintercept=state_median, linetype="dashed", linewidth=0.5, color="grey40") +
  annotate("text", x=state_median, y=Inf, label=median_label,
           size=3, color="grey30", hjust=0.5, vjust=-0.5) +
  geom_segment(aes(x=state_median, xend=mean_pp, yend=state_label), linewidth=0.9, alpha=0.5) +
  geom_point(size=2.6) +
  scale_color_manual(values=c("Below Average"="#EB6834", "Above Average"="#1BAF7A")) +
  scale_x_continuous(labels=label_dollar(scale=1e-3, suffix="k")) +
  coord_cartesian(clip="off") +
  labs(x="Mean Purchasing Power", y=NULL) +
  theme(axis.text.y=element_text(size=7), plot.margin=margin(20,10,10,10))

### Collinearity

The reference group logic of <a href="#fig-vif" class="quarto-xref">Figure 5</a> applies here unchanged, so `non_routine_cognitive_share` is again omitted and the remaining three coefficients read as shifts toward each group and away from it. What is new in this feature set is the indicator blocks. <a href="#fig-vif-ml" class="quarto-xref">Figure 12</a> checks that the year and state indicators introduce no collinearity of their own.

In [30]:
exposure_check_cols=["routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share"]

REFERENCE_EXPOSURE="non_routine_cognitive_share"
exposure_cols_model=[c for c in exposure_check_cols if c!=REFERENCE_EXPOSURE]
core_features=exposure_cols_model+["poverty_rate", "unemployment_rate"]

In [31]:
vif_features=core_features+year_cols_model+state_cols_model

X_vif=sm.add_constant(model_df[vif_features].astype(float))

vif_data=pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_top=vif_data[vif_data["feature"].isin(year_cols_model+state_cols_model)].sort_values("VIF", ascending=False).head(10).copy()
vif_top["Indicator"]=(vif_top["feature"]
                      .str.replace("year_", "Year ", regex=False)
                      .str.replace("state_", "State ", regex=False))
vif_plot_df=vif_top[["Indicator","VIF"]].sort_values("VIF")

In [32]:
%%R -i vif_plot_df -w 8 -h 4 -u in -r 150
vif_plot_df$Indicator <- factor(vif_plot_df$Indicator, levels=vif_plot_df$Indicator)

ggplot(vif_plot_df, aes(x=VIF, y=Indicator)) +
  geom_segment(aes(x=0, xend=VIF, yend=Indicator), linewidth=1.1, color="#C8C8C5") +
  geom_point(size=4, color="#1E8E99") +
  geom_text(aes(label=sprintf("%.2f", VIF)), hjust=-0.5, size=3.6, color="#252525") +
  geom_vline(xintercept=5, linetype="dashed", linewidth=0.7, color="#EB6834") +
  annotate("text", x=5, y=Inf, label="VIF = 5", hjust=1.1, vjust=1.3, color="#993F00", size=3.5) +
  scale_x_continuous(limits=c(0, 5.6), breaks=0:5, expand=c(0,0)) +
  labs(x="Variance Inflation Factor", y=NULL)

<a href="#fig-vif-ml" class="quarto-xref">Figure 12</a> lists the highest variance inflation factors after the reference group is removed. No offenders remain, since every feature falls comfortably below the threshold of 5.

### Grouped train and test split

This is not a forecasting exercise, so a time ordered split is not required for its usual reason. A grouped split is still necessary, because a county’s economic profile barely moves year to year, which makes its 2018 and 2019 rows near duplicates. If two adjacent rows land on opposite sides of the split, the model can recall a county rather than learn a relationship. We therefore group by `county_fips` when splitting, so every test set prediction comes from a county the model has seen no rows of, which tests whether the pattern generalizes rather than whether it was memorized.

In [33]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

feature_cols_reg=core_features+year_cols_model+state_cols_model
target_col="affordability_salary"

X_ml=model_df[feature_cols_reg].reset_index(drop=True).astype(float)
y=model_df[target_col].reset_index(drop=True)
groups=model_df["county_fips"].reset_index(drop=True)

gss=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx=next(gss.split(X_ml, y, groups))

X_train, X_test=X_ml.iloc[train_idx], X_ml.iloc[test_idx]
y_train, y_test=y.iloc[train_idx], y.iloc[test_idx]
groups_train=groups.iloc[train_idx]

assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))==0

In [34]:
# baseline feature set (no year/state), reusing the SAME row split as the full model
feature_cols_base=exposure_cols_model+["poverty_rate", "unemployment_rate"]
X_base=model_df[feature_cols_base].reset_index(drop=True).astype(float)
X_train_base, X_test_base=X_base.iloc[train_idx], X_base.iloc[test_idx]

### Linear regression

This specification is a predictive benchmark, not a second inferential model. It reuses the diagnostic lessons the panel regression above already established, now fit on the training split alone so it is directly comparable to the random forest and neural network that follow. We report two versions of each linear model, estimated by ordinary least squares (OLS). The baseline uses the task groups with poverty and unemployment only, and the full model adds year and state controls. The baseline shows what the relationship looks like before time and geography are accounted for, the full model is the one carried forward, and reporting both makes visible exactly what changes when time and geography enter.

#### Baseline specification

In [35]:
X_train_base_sm=sm.add_constant(X_train_base)
ols_model_base=sm.OLS(y_train, X_train_base_sm).fit()

The baseline reaches an R² of 0.766. Every coefficient is negative, which follows from non-routine cognitive serving as the reference. Shifting a county toward any of the other three groups is associated with lower purchasing power, and non-routine manual is the steepest. The coefficients appear alongside the full specifications in <a href="#tbl-ols-comparison" class="quarto-xref">Table 2</a>.

#### Full specification

In [36]:
X_train_sm=sm.add_constant(X_train)
ols_model=sm.OLS(y_train, X_train_sm).fit()

Adding time and geography raises R² from 0.766 to 0.871, so these variables explain real variance. The `unemployment_rate` coefficient flips sign, which suggests the baseline coefficient absorbed some of the geographic and temporal confounding, and that is what we would expect, since places and periods with heavy unemployment also tend to be places and periods of low purchasing power. The full model’s coefficient reflects the within state year relationship instead. The task group coefficients keep their direction and their relative ordering through the addition.

The raw target residuals show the same systematic curvature already documented in <a href="#fig-diagnostics" class="quarto-xref">Figure 8</a>, so the linear specification is poorly suited to the untransformed target here as well.

#### Full specification on the logged outcome

The binned means in <a href="#fig-binned-scale" class="quarto-xref">Figure 9</a> showed an approximately exponential pattern, so we test whether logging the target makes a linear model appropriate here as well.

In [37]:
assert (y<=0).sum()==0  # confirm log is safe

y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

ols_model_log=sm.OLS(y_train_log, X_train_sm).fit()

X_test_sm=sm.add_constant(X_test, has_constant="add")
y_pred_ols_log=ols_model_log.predict(X_test_sm)
y_pred_ols_log_dollars=np.exp(y_pred_ols_log)

ols_log_test_r2_log=r2_score(y_test_log, y_pred_ols_log)
ols_log_test_r2=r2_score(y_test, y_pred_ols_log_dollars)
ols_log_test_mae=mean_absolute_error(y_test, y_pred_ols_log_dollars)

Logging the target resolves the pattern here the same way it did for the panel regression in <a href="#fig-diagnostics" class="quarto-xref">Figure 8</a>: the residual spread evens out and the QQ plot tracks the diagonal more closely, so a linear model is appropriate on the logged target. That model reaches an in sample R² of 0.913 and a held out R², back transformed into dollars, of 0.896.

#### Coefficient comparison

<a href="#tbl-ols-comparison" class="quarto-xref">Table 2</a> reports the coefficients across the three specifications, and <a href="#sec-results" class="quarto-xref">Section 5</a> shows the same comparison graphically.

In [38]:
log_coefs=ols_model_log.params[feature_cols_base]

# task groups are 0 to 1 scale (1pp=0.01 units); poverty and unemployment are already in point units
unit_per_point={
    "routine_cognitive_share": 0.01,
    "routine_manual_share": 0.01,
    "non_routine_manual_share": 0.01,
    "poverty_rate": 1.0,
    "unemployment_rate": 1.0,
}

pct_change_per_point={
    feat: (np.exp(log_coefs[feat]*unit_per_point[feat])-1)*100
    for feat in feature_cols_base
}

label_map={
    "routine_cognitive_share":"Routine cognitive",
    "routine_manual_share":"Routine manual",
    "non_routine_manual_share":"Non-routine manual",
    "poverty_rate":"Poverty rate",
    "unemployment_rate":"Unemployment rate",
}

comparison_ols=pd.DataFrame({
    "Variable": [label_map[f] for f in feature_cols_base],
    "Baseline ($)": ols_model_base.params[feature_cols_base].values,
    "With controls ($)": ols_model.params[feature_cols_base].values,
    "With controls (log)": ols_model_log.params[feature_cols_base].values,
    "Percent change per point": [pct_change_per_point[f] for f in feature_cols_base],
})

(GT(comparison_ols)
  .fmt_number(columns=["Baseline ($)","With controls ($)"], decimals=0, use_seps=True)
  .fmt_number(columns="With controls (log)", decimals=4)
  .fmt_number(columns="Percent change per point", decimals=2))

The sign flip in `unemployment_rate` holds across all three fitted forms. Task group coefficients shrink in magnitude from baseline to raw controls while the poverty coefficient grows, from roughly −1,376 to −1,729, so the controls sharpen the task group association rather than displacing it.

Each additional percentage point of poverty rate is associated with roughly 2.9 percent lower purchasing power, holding time, geography, and task groups fixed. That is a considerably larger proportional difference than any single task group carries.

In [39]:
model_summary = pd.DataFrame([
    {"model": "Baseline", "metric": "Outcome", "value": "Purchasing power ($)", "status": "neutral"},
    {"model": "Full (raw $)", "metric": "Outcome", "value": "Purchasing power ($)", "status": "neutral"},
    {"model": "Full (log)", "metric": "Outcome", "value": "log(Purchasing power)", "status": "neutral"},
    {"model": "Baseline", "metric": "Year + state FE", "value": "No", "status": "neutral"},
    {"model": "Full (raw $)", "metric": "Year + state FE", "value": "Yes", "status": "neutral"},
    {"model": "Full (log)", "metric": "Year + state FE", "value": "Yes", "status": "neutral"},
    {"model": "Baseline", "metric": "R squared", "value": f"{ols_model_base.rsquared:.3f}", "status": "neutral"},
    {"model": "Full (raw $)", "metric": "R squared", "value": f"{ols_model.rsquared:.3f}", "status": "neutral"},
    {"model": "Full (log)", "metric": "R squared", "value": f"{ols_model_log.rsquared:.3f}", "status": "neutral"},
    {"model": "Baseline", "metric": "Diagnostics", "value": "Not assessed", "status": "neutral"},
    {"model": "Full (raw $)", "metric": "Diagnostics", "value": "Failed", "status": "fail"},
    {"model": "Full (log)", "metric": "Diagnostics", "value": "Passed", "status": "success"},
    {"model": "Baseline", "metric": "Decision", "value": "Superseded", "status": "fail"},
    {"model": "Full (raw $)", "metric": "Decision", "value": "Not used", "status": "fail"},
    {"model": "Full (log)", "metric": "Decision", "value": "Used for inference", "status": "success"},
])

In [40]:
%%R -i model_summary -w 7.5 -h 4.2 -u in -r 150
model_summary$metric <- factor(model_summary$metric,
  levels=rev(c("Outcome", "Year + state FE", "R squared", "Diagnostics", "Decision")))
model_summary$model <- factor(model_summary$model,
  levels=c("Baseline", "Full (raw $)", "Full (log)"))

ggplot(model_summary, aes(x=model, y=metric, fill=status)) +
  geom_tile(color="#FCFCFB", linewidth=3, width=0.98, height=0.94) +
  geom_text(aes(label=value, fontface=ifelse(metric %in% c("R squared", "Decision"), "bold", "plain")),
            size=3.3, color="#252525") +
  scale_fill_manual(values=c(neutral="#F1F1EE", fail="#EB6834", success="#1E8E99"), guide="none") +
  scale_x_discrete(position="top") +
  labs(x=NULL, y=NULL) +
  theme(panel.grid=element_blank(), axis.ticks=element_blank())

<a href="#fig-specmatrix" class="quarto-xref">Figure 13</a> shows in sample R² climbing through each stage. Adding year and state controls raises it from 0.766 to 0.871, confirming that time and geography explain real variance the baseline left unmodeled. Logging the target raises it further, to 0.913, and resolves the residual violations along the way. This is the linear model used for inference from here, since the raw target model’s coefficient significance cannot be trusted given its failed diagnostics.

### Random forest

For the random forest we ran a grid search over the number of trees (600 and 1,000), maximum depth (10, 20, and unlimited), minimum leaf size (1, 2, and 5), and the feature subsampling rate (all features, half, and the square root), scored by grouped five fold cross validation on the training counties. More trees reduce variance, while depth and leaf size supply regularization. The search selected 1,000 trees, unlimited depth, a minimum leaf of 1, and half the features per split.

In [41]:
# grid search actually run once; kept for documentation of the search space,
# not re-executed on render. Best params hardcoded in the following chunk.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold

rf=RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid={
    "n_estimators": [600, 1000],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [1.0, "sqrt", 0.5],
}

group_cv=GroupKFold(n_splits=5)
gcv=GridSearchCV(rf, param_grid, cv=group_cv, scoring="r2", n_jobs=-1)
gcv.fit(X_train, y_train, groups=groups_train)

In [42]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# grid search above was run once over the full param_grid; best params found were:
# {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 1000}
best_params={
    "max_depth": None,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "n_estimators": 1000,
}

best_rf=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
best_rf.fit(X_train, y_train)

y_pred_rf=best_rf.predict(X_test)
rf_test_r2=r2_score(y_test, y_pred_rf)
rf_test_mae=mean_absolute_error(y_test, y_pred_rf)
rf_train_r2=best_rf.score(X_train, y_train)

The forest reaches a held out R² of 0.876 on the raw target. The log target OLS model’s held out figure of 0.896 is the relevant comparison, since both are scored on the same held out counties, and the forest does not improve on it.

The forest fits the training data far more closely than it fits unseen counties, 0.989 against 0.876. Its additional flexibility therefore captures patterns in the training sample that do not carry over to counties it has not seen.

In [43]:
rf_resid_df=pd.DataFrame({"pred":y_pred_rf, "resid":y_test-y_pred_rf})

In [44]:
%%R -i rf_resid_df -w 7 -h 5 -u in -r 150
ggplot(rf_resid_df, aes(x=pred, y=resid)) +
  geom_point(alpha=0.3, size=0.6, color="#CC79A7") +
  geom_hline(yintercept=0, linetype="dashed", linewidth=0.4) +
  labs(x="Predicted Values ($)", y="Residuals ($)")

Unlike the raw target OLS residuals, <a href="#fig-rf-resid" class="quarto-xref">Figure 14</a> shows no systematic curve or trend across the range of predicted values.

In [45]:
perm_imp=permutation_importance(
    best_rf, X_test, y_test, n_repeats=20, random_state=42, n_jobs=1
)

#### Random forest on the logged outcome

In [46]:
rf_log=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
rf_log.fit(X_train, y_train_log)

y_pred_rf_log=rf_log.predict(X_test)
y_pred_rf_log_dollars=np.exp(y_pred_rf_log)

rf_log_test_r2_log=r2_score(y_test_log, y_pred_rf_log)
rf_log_test_r2=r2_score(y_test, y_pred_rf_log_dollars)
rf_log_test_mae=mean_absolute_error(y_test, y_pred_rf_log_dollars)

Fitting the forest on the logged outcome gives 0.877 after back transforming, a negligible change from 0.876 on the raw target. Since the log target is already the reference form for the linear model, we adopt the log target forest for the comparisons that follow, even though the choice of target makes little practical difference for the forest.

#### Ablation without task groups

In [47]:
feature_cols_no_exposure=[c for c in feature_cols_reg if c not in exposure_cols_model]
X_train_ne=X_train[feature_cols_no_exposure]
X_test_ne=X_test[feature_cols_no_exposure]

rf_no_exposure=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_exposure.fit(X_train_ne, y_train_log)
rf_no_exposure_r2=r2_score(y_test, np.exp(rf_no_exposure.predict(X_test_ne)))

Without the task groups, the same random forest reaches an R² of 0.834.

#### Ablation without economic controls

In [48]:
feature_cols_no_econ=[c for c in feature_cols_reg if c not in ["poverty_rate", "unemployment_rate"]]
X_train_ne2=X_train[feature_cols_no_econ]
X_test_ne2=X_test[feature_cols_no_econ]

rf_no_econ=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_econ.fit(X_train_ne2, y_train_log)
rf_no_econ_r2=r2_score(y_test, np.exp(rf_no_econ.predict(X_test_ne2)))

Using only the task groups, with poverty and unemployment removed, the model reaches 0.703. That is meaningful on its own but well short of the full model. All three figures come from the same forest specification fit on the log target and scored on the same held out counties, so they sit on one scale.

#### Grouped permutation

To see what the task groups do together, rather than one at a time diluting each other, we permute the modeled task group proportions jointly and compare the drop in R² against permuting them individually.

In [49]:
exposure_cols=exposure_cols_model  # the three non-reference task groups
rng=np.random.RandomState(42)

baseline_r2=r2_score(y_test, best_rf.predict(X_test))

def grouped_permutation_drop(cols, n_repeats=20):
    drops=[]
    for _ in range(n_repeats):
        X_perm=X_test.copy()
        shuffled_idx=rng.permutation(len(X_perm))
        X_perm[cols]=X_perm[cols].values[shuffled_idx]
        drops.append(baseline_r2-r2_score(y_test, best_rf.predict(X_perm)))
    return np.mean(drops)

joint_drop=grouped_permutation_drop(exposure_cols)

individual_drops={}
for col in exposure_cols:
    individual_drops[col]=grouped_permutation_drop([col])

year_drop=grouped_permutation_drop(year_cols_model)
state_drop=grouped_permutation_drop(state_cols_model)

Year matters far more than state, at 0.128 against 0.008, so time rather than geography carries the bulk of the temporal and spatial signal once the economic variables and task groups are present.

The task group proportions drop 0.153 when permuted jointly, considerably more than the ablation above implies. They therefore overlap with poverty, unemployment, and time while still adding something those three do not capture.

Among the three modeled proportions, `non_routine_manual_share` shows the largest sensitivity, though none should be read as independently additive given that they are parts of a whole. This matches the panel regression, where non-routine manual carries the steepest coefficient.

### Neural network

The neural network is kept shallow, with three hidden layers and a single linear output unit for the continuous outcome.

In [50]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

scaler_x=StandardScaler().fit(X_train)
X_train_s=scaler_x.transform(X_train)
X_test_s=scaler_x.transform(X_test)

scaler_y=StandardScaler().fit(y_train.values.reshape(-1, 1))
y_train_s=scaler_y.transform(y_train.values.reshape(-1, 1)).ravel()
y_test_s=scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

n_features=X_train.shape[1]
h1=int(round(n_features*1.5))
h2=max(int(round(h1*0.5)), 20)
h3=10

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h2, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),  # regression head
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=15, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_initial_test_r2=r2_score(y_test, y_pred_nn)
nn_initial_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_init_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

Training loss falls steadily while validation loss reverses direction partway through, which is overfitting; <a href="#fig-nn-curves" class="quarto-xref">Figure 15</a> shows this alongside the regularized model below. We respond with L2 weight regularization, heavier dropout, and a shorter early stopping patience.

In [51]:
from tensorflow.keras import regularizers

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(h2, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=8, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_test_r2=r2_score(y_test, y_pred_nn)
nn_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_reg_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

In [52]:
%%R -i nn_init_df -i nn_reg_df -w 10 -h 4.5 -u in -r 150
TEAL <- "#1E8E99"; ORANGE <- "#FF8E32"

ymax <- max(c(nn_init_df$loss, nn_reg_df$loss)) * 1.05

best_init <- nn_init_df[nn_init_df$series=="Validation",]
best_init_epoch <- best_init$epoch[which.min(best_init$loss)]
best_reg <- nn_reg_df[nn_reg_df$series=="Validation",]
best_reg_epoch <- best_reg$epoch[which.min(best_reg$loss)]

label_df1 <- nn_init_df[nn_init_df$epoch==max(nn_init_df$epoch),]
label_df2 <- nn_reg_df[nn_reg_df$epoch==max(nn_reg_df$epoch),]

p1 <- ggplot(nn_init_df, aes(x=epoch, y=loss, color=series)) +
  annotate("rect", xmin=best_init_epoch, xmax=max(nn_init_df$epoch), ymin=-Inf, ymax=Inf,
           fill=ORANGE, alpha=0.08) +
  geom_vline(xintercept=best_init_epoch, linetype="dashed", linewidth=0.5, color="grey50") +
  geom_line(linewidth=0.8) +
  geom_text(data=label_df1, aes(label=series), hjust=-0.1, size=3.4, fontface="bold", show.legend=FALSE) +
  annotate("text", x=best_init_epoch, y=ymax*0.94, label=paste0("Best epoch = ", best_init_epoch),
           hjust=-0.1, size=3.2, color="grey40") +
  annotate("text", x=max(nn_init_df$epoch)*0.72, y=ymax*0.5, label="Overfitting begins here",
           size=3.3, color="#B35A00", fontface="italic") +
  scale_color_manual(values=c(Train=TEAL, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax)) +
  scale_x_continuous(expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y="Loss (MSE, Standardized)", title="Initial Model",
       subtitle="Light dropout, patience 15")

p2 <- ggplot(nn_reg_df, aes(x=epoch, y=loss, color=series)) +
  geom_vline(xintercept=best_reg_epoch, linetype="dashed", linewidth=0.5, color="grey50") +
  geom_line(linewidth=0.8) +
  geom_text(data=label_df2, aes(label=series, y=loss + ifelse(series=="Train", -0.025, 0.025)),
            hjust=-0.1, size=3.4, fontface="bold", show.legend=FALSE) +
  annotate("text", x=best_reg_epoch, y=ymax*0.94, label=paste0("Best epoch = ", best_reg_epoch),
           hjust=1.1, size=3.2, color="grey40") +
  annotate("text", x=max(nn_reg_df$epoch)*0.5, y=ymax*0.5, label="Improved generalization",
           size=3.3, color=TEAL, fontface="italic") +
  scale_color_manual(values=c(Train=TEAL, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax)) +
  scale_x_continuous(expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y=NULL, title="Tuned Model",
       subtitle="L2 regularization, heavier dropout, patience 8")

p1 + p2

The gap narrows after regularization, but the network still does not beat the random forest or the log target linear model, both of which are far cheaper to fit and tune. This project is built to explain the relationship between task groups and purchasing power rather than to maximize predictive accuracy, so we stop tuning here and carry the linear and forest models forward as the primary results.

### Cross validation

Having compared explained variance across the three models, we cross validate to check that the single split result generalizes.

In [53]:
from sklearn.model_selection import GroupKFold

group_kfold_cv=GroupKFold(n_splits=5)
cv_results={
    "OLS (log target)": [],
    "Random forest (log target)": [],
}

y_log=np.log(y)  # full data log target, same idea as y_train_log/y_test_log

for fold, (tr_idx, val_idx) in enumerate(group_kfold_cv.split(X_ml, y, groups)):
    X_tr, X_val=X_ml.iloc[tr_idx], X_ml.iloc[val_idx]
    y_tr, y_val=y.iloc[tr_idx], y.iloc[val_idx]
    y_tr_log=y_log.iloc[tr_idx]

    X_tr_sm=sm.add_constant(X_tr)
    X_val_sm=sm.add_constant(X_val, has_constant="add")
    ols_fold_log=sm.OLS(y_tr_log, X_tr_sm).fit()
    pred_ols=np.exp(ols_fold_log.predict(X_val_sm))  # back transformed, dollar scale
    cv_results["OLS (log target)"].append(r2_score(y_val, pred_ols))

    rf_fold=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr_log)
    pred_rf=np.exp(rf_fold.predict(X_val))
    cv_results["Random forest (log target)"].append(r2_score(y_val, pred_rf))

cv_folds=pd.DataFrame([
    {"model": m, "fold": i+1, "r2": s}
    for m, scores in cv_results.items()
    for i, s in enumerate(scores)
])

cv_summary=pd.DataFrame({m: {"Mean R squared": np.mean(s), "SD": np.std(s)}
                         for m, s in cv_results.items()}).T.reset_index()
cv_summary.columns=["Model","Mean R squared","SD"]

In [54]:
(GT(cv_summary)
  .fmt_number(columns=["Mean R squared","SD"], decimals=4)
  .cols_align(align="right", columns=["Mean R squared","SD"]))

<a href="#tbl-cv-summary" class="quarto-xref">Table 3</a> shows the two models performing at the same level across five folds. The difference between their means is smaller than the variation across folds, so the models perform similarly relative to that variation. The single split result above is therefore not an artifact of one particular split, and both models generalize comparably across held out counties.

The neural network is not included, since cross validating it would be expensive and it is not carried forward.

<a href="#sec-results" class="quarto-xref">Section 5</a> reports the model comparison across held out R² and mean absolute error.

## Summary

Three specifications answer three different questions. The panel regression, with standard errors clustered by county, is the model used for inference. It shows that routine intensive work is associated with lower purchasing power even after controlling for poverty, unemployment, population, and year, and the variance decomposition establishes that the task group differences behind that association are durable features of a place rather than transient ones. Its log respecification resolves the residual violations that make the raw target version untrustworthy for that purpose.

The predictive models answer a different question, which is how much of purchasing power can be explained at all, and by what. The random forest and the log target linear model perform comparably on held out data, so the curvature in the relationship adds little once poverty, unemployment, time, and geography are already in the model. The neural network improves on neither, and its added complexity is not justified by this feature set.

Across both approaches the task groups carry real, independent explanatory power, while poverty rate remains the dominant single factor. That ordering, rather than any one model’s fit statistic, is the finding this analysis is built to support, and it is what <a href="#sec-results" class="quarto-xref">Section 5</a> carries forward.

# Results

In [55]:
# inbound from _04: reg, df, model, model_log, X_ml, X_test, y_test, y_pred_rf,
# y_pred_rf_log_dollars, best_rf, perm_imp, joint_drop, year_drop, state_drop,
# cv_folds, engine, ols_log_test_r2, rf_log_test_r2, nn_test_r2,
# ols_log_test_mae, rf_log_test_mae, nn_test_mae
import pandas as pd
import numpy as np
import statsmodels.api as sm
from great_tables import GT

res_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

mean_pp=float(reg["affordability_salary"].mean())
group_terms=["non_routine_manual_share","routine_cognitive_share","routine_manual_share"]
group_names=["Non-Routine Manual","Routine Cognitive","Routine Manual"]

The analysis supports four findings. The balance of work in a county is associated with its purchasing power, and the association survives controls for poverty, unemployment, population, and year. The log specification captures most of the structure in the relationship, and the flexible models do not improve on it, which is what supports reading the relationship as proportional rather than fixed in dollars. Poverty rate remains the single strongest correlate of purchasing power throughout, with the task groups adding real signal beyond it. And the task group differences behind the association are durable features of places, so the purchasing power gaps they carry are durable too. The subsections below work through these findings, and through the geography and the model behavior behind them.

## The association

In [56]:
# log specification coefficients (model_log fit in _04, cluster robust), expressed as
# dollars at the panel mean per one percentage point shift
coef_rows=[]
for term, name in zip(group_terms, group_names):
    b=model_log.params[term]
    se=model_log.bse[term]
    est=(np.exp(b*0.01)-1)*mean_pp
    lo=(np.exp((b-1.96*se)*0.01)-1)*mean_pp
    hi=(np.exp((b+1.96*se)*0.01)-1)*mean_pp
    coef_rows.append({"group":name, "est":est, "lo":lo, "hi":hi})
coef_df=pd.DataFrame(coef_rows)

The log specification of the panel regression is the model we use for inference, for the diagnostic reasons set out in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. <a href="#fig-coefplot" class="quarto-xref">Figure 16</a> reports its task group coefficients as the change in purchasing power, in dollars at the panel mean, associated with a one percentage point shift into each group and out of non-routine cognitive work, the reference group. A one percentage point shift toward non-routine manual work is associated with roughly \$846 less in purchasing power, with a 95 percent interval of plus or minus \$38, evaluated at the panel mean. Routine cognitive and routine manual shifts carry smaller but clearly negative associations, and the ordering matches the level specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. For scale, each additional percentage point of poverty rate is associated with roughly 2.9 percent lower purchasing power, a larger proportional difference than any single task group carries.

In [57]:
%%R -i coef_df -w 8 -h 3.5 -u in -r 150
coef_df$group <- factor(coef_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))

ggplot(coef_df, aes(x=est, y=group)) +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.4) +
  geom_errorbar(aes(xmin=lo, xmax=hi), orientation="y", width=0.15, color="#0072B2") +
  geom_point(size=2.6, color="#0072B2") +
  scale_x_continuous(labels=dollar_axis) +
  labs(x="Change in Purchasing Power per Percentage Point Shift ($ at Panel Mean)", y=NULL)

## Not poverty in disguise

A natural objection is that the task groups merely proxy for poverty, since poor counties hold more routine work. <a href="#fig-baseline-maps" class="quarto-xref">Figure 17</a> shows the two conventional distress measures side by side, mean poverty rate and mean unemployment rate by county across the study period. Both run highest across the rural South, the Mississippi Delta, and pockets of the Southwest border region, the same broad area where <a href="#fig-afford-map" class="quarto-xref">Figure 19</a> shows purchasing power running lowest, which is exactly why the objection is worth taking seriously. Unemployment carries a second concentration along the California coast and Central Valley that poverty does not share, evidence the two measures are not interchangeable with each other or with what follows.

In [58]:
import geopandas as gpd

map_baseline=pd.read_sql("""
    select county_fips, avg(poverty_rate) as poverty_rate, avg(unemployment_rate) as unemployment_rate
    from county_baseline
    group by county_fips
""", engine)
map_baseline['fips']=map_baseline['county_fips'].astype(str).str.zfill(5)

gdf_baseline=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_baseline=gdf_baseline.rename(columns={'GEOID':'fips'})
merged_baseline=gdf_baseline.merge(map_baseline[['fips','poverty_rate','unemployment_rate']], on='fips', how='left')
merged_baseline=merged_baseline[~merged_baseline['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_baseline[['fips','poverty_rate','unemployment_rate','geometry']].to_file("output/baseline_maps.geojson", driver="GeoJSON")

pov_vmin=float(merged_baseline['poverty_rate'].quantile(0.02))
pov_vmax=float(merged_baseline['poverty_rate'].quantile(0.98))
unemp_vmin=float(merged_baseline['unemployment_rate'].quantile(0.02))
unemp_vmax=float(merged_baseline['unemployment_rate'].quantile(0.98))

In [59]:
%%R -i pov_vmin -i pov_vmax -i unemp_vmin -i unemp_vmax -w 11 -h 5.5 -u in -r 150
suppressMessages(library(sf))

baseline_sf <- st_read("output/baseline_maps.geojson", quiet=TRUE)

p1 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(pov_vmin, pov_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Poverty Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill="#FCFCFB", color=NA)) +
  labs(title="Poverty Rate")

p2 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(unemp_vmin, unemp_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Unemployment Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill="#FCFCFB", color=NA)) +
  labs(title="Unemployment Rate")

p1 + p2

<a href="#fig-coefcompare" class="quarto-xref">Figure 18</a> tests the objection directly by fitting the log panel regression twice on the same rows, once with task groups and year indicators only, and once adding poverty rate, unemployment rate, and log population, the specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. Non-routine manual and routine manual both shrink substantially when the controls enter, which is what we would expect, since poverty absorbs shared variation. Routine cognitive barely moves, at roughly −1.1 percent either way, evidence its association runs through something other than the poverty and unemployment channel the controls capture. All three remain negative and precisely estimated after the controls enter. The task group association is smaller than it first appears for two of the three groups and clearly real for all of them.

In [60]:
share_terms=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

X_nocontrols=pd.concat([reg[share_terms],
                        pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_nocontrols=sm.add_constant(X_nocontrols)
model_nc_log=sm.OLS(reg["actual_log"], X_nocontrols).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

def pct_ci(m, term):
    b=m.params[term]
    se=m.bse[term]
    est=(np.exp(b*0.01)-1)*100
    lo=(np.exp((b-1.96*se)*0.01)-1)*100
    hi=(np.exp((b+1.96*se)*0.01)-1)*100
    return est, lo, hi

compare_rows=[]
for term, name in zip(share_terms, ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    for m, spec in [(model_nc_log, "Task Groups and Year Only"), (model_log, "With Controls")]:
        est, lo, hi = pct_ci(m, term)
        compare_rows.append({"group":name, "spec":spec, "est":est, "lo":lo, "hi":hi})
compare_df=pd.DataFrame(compare_rows)

compare_wide=compare_df.pivot(index="group", columns="spec", values="est").reset_index()
compare_wide.columns=["group","est_baseline","est_controls"]
compare_wide["attenuation_pct"]=100*(1-compare_wide["est_controls"].abs()/compare_wide["est_baseline"].abs())
compare_wide["attenuation_label"]=compare_wide["attenuation_pct"].apply(
    lambda v: "Little change" if v<5 else f"{v:.0f}% smaller")

In [61]:
%%R -i compare_df -i compare_wide -w 9 -h 4.5 -u in -r 150
TEAL <- "#1E8E99"; ORANGE <- "#FF8E32"
compare_df$group <- factor(compare_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))
compare_df$spec <- factor(compare_df$spec,
    levels=c("Task Groups and Year Only","With Controls"))
compare_wide$group <- factor(compare_wide$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))
compare_df$label_nudge <- ifelse(compare_df$spec=="With Controls", 0.35, -0.35)
compare_wide$mid <- (compare_wide$est_baseline+compare_wide$est_controls)/2

ggplot(compare_df, aes(x=est, y=group)) +
  geom_segment(data=compare_wide, aes(x=est_baseline, xend=est_controls, y=group, yend=group),
               inherit.aes=FALSE, color="grey75", linewidth=2.0, lineend="round") +
  geom_errorbar(aes(xmin=lo, xmax=hi, color=spec), orientation="y", width=0.1, linewidth=0.6) +
  geom_point(aes(color=spec, size=spec)) +
  geom_text(aes(label=sprintf("%.1f%%", est), color=spec, y=as.numeric(group)+label_nudge),
            size=3.3, show.legend=FALSE, fontface="bold") +
  geom_text(data=compare_wide, aes(x=mid, y=as.numeric(group)-0.32, label=attenuation_label),
            inherit.aes=FALSE, color="grey40", size=3.0, fontface="italic") +
  scale_size_manual(values=c("Task Groups and Year Only"=2.6, "With Controls"=3.6), guide="none") +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.4) +
  scale_color_manual(values=c("Task Groups and Year Only"=ORANGE, "With Controls"=TEAL)) +
  scale_y_discrete(expand=expansion(add=0.7)) +
  labs(x="Percent Change in Purchasing Power per Percentage Point Shift", y=NULL, color=NULL) +
  theme(legend.position="bottom")

## Where purchasing power is strained

The association has a geography. <a href="#fig-afford-map" class="quarto-xref">Figure 19</a> maps mean purchasing power by county across the study period, on the full set of counties with a purchasing power value rather than the analytical panel alone, since the outcome requires only income and a price parity. Purchasing power runs highest across the metropolitan Northeast corridor, the upper Midwest, and pockets of the mountain West, and lowest across the rural South and the southern border region. The bottom panel carries the point further: splitting counties by whichever task group holds the largest share of local employment, the 664 counties where non-routine cognitive work dominates reach substantially higher and more varied purchasing power than the 97 dominated by non-routine manual work or the 87 dominated by routine manual work, which both cluster tightly at the low end. No county in the panel has routine cognitive work as its largest group, so that curve does not appear. The map shows the outcome side of the association the coefficients estimate, and the density panel shows the same association holding at the level of a county’s single largest task group.

In [62]:
import geopandas as gpd

map_all=pd.read_sql("""
    select county_fips, avg(affordability_salary) as purchasing_power
    from county_affordability
    group by county_fips
""", engine)
map_all['fips']=map_all['county_fips'].astype(str).str.zfill(5)

gdf_map=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_map=gdf_map.rename(columns={'GEOID':'fips'})

afford_vmin=float(map_all['purchasing_power'].quantile(0.02))
afford_vmax=float(map_all['purchasing_power'].quantile(0.98))

merged_map=gdf_map.merge(map_all[['fips','purchasing_power']], on='fips', how='left')
merged_map=merged_map[~merged_map['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_map[['fips','purchasing_power','geometry']].to_file("output/afford_map.geojson", driver="GeoJSON")

# dominant task group per county: whichever of the four groups holds the largest mean share
dominant_share_cols=["routine_cognitive_share","routine_manual_share",
                     "non_routine_cognitive_share","non_routine_manual_share"]
dominant_label_map={"routine_cognitive_share":"Routine Cognitive","routine_manual_share":"Routine Manual",
                    "non_routine_cognitive_share":"Non-Routine Cognitive","non_routine_manual_share":"Non-Routine Manual"}
dominant_df=pd.read_sql(f"""
    select cte.county_fips,
           avg(cte.routine_cognitive_share) as routine_cognitive_share,
           avg(cte.routine_manual_share) as routine_manual_share,
           avg(cte.non_routine_cognitive_share) as non_routine_cognitive_share,
           avg(cte.non_routine_manual_share) as non_routine_manual_share,
           avg(ca.affordability_salary) as purchasing_power
    from county_task_exposure cte
    join county_affordability ca on cte.county_fips=ca.county_fips and cte.year=ca.year
    group by cte.county_fips
""", engine)
dominant_df["dominant_group"]=dominant_df[dominant_share_cols].idxmax(axis=1).map(dominant_label_map)

In [63]:
%%R -i afford_vmin -i afford_vmax -i dominant_df -w 9 -h 9 -u in -r 150
suppressMessages(library(sf))

afford_sf <- st_read("output/afford_map.geojson", quiet=TRUE)

purchasing_power_colors <- c("#E5FFFF","#CCFEFF","#8EDDE2","#51C3CC","#1E8E99","#007A7A","#006666")
DOMINANT_COLORS <- c("Non-Routine Cognitive"="#FFAD65", "Non-Routine Manual"="#993F00", "Routine Manual"="#006666")

p1 <- ggplot(afford_sf) +
  geom_sf(aes(fill=purchasing_power), color="#FAFAF8", linewidth=0.05) +
  scale_fill_gradientn(colors=purchasing_power_colors, limits=c(afford_vmin, afford_vmax),
                        oob=scales::squish, na.value="#D9D9D6",
                        labels=label_dollar(scale=1e-3, suffix="k"),
                        name="Mean Purchasing Power") +
  coord_sf(crs=st_crs(5070), datum=NA) +
  guides(fill=guide_colorbar(direction="horizontal", title.position="top", title.hjust=0.5,
                             barwidth=unit(4.5,"cm"), barheight=unit(0.35,"cm"))) +
  theme_void() +
  theme(legend.position="bottom",
        legend.title=element_text(size=10, face="bold", color="#333333"),
        legend.text=element_text(size=9, color="#555555"),
        plot.background=element_rect(fill="#FCFCFB", color=NA))

p2 <- ggplot(dominant_df, aes(x=purchasing_power, fill=dominant_group, color=dominant_group)) +
  geom_density(alpha=0.35, linewidth=0.7) +
  scale_fill_manual(values=DOMINANT_COLORS) +
  scale_color_manual(values=DOMINANT_COLORS) +
  scale_x_continuous(labels=dollar_axis) +
  labs(x="Mean Purchasing Power ($)", y="Density", fill=NULL, color=NULL)

(p1 / p2) + plot_layout(heights=c(2.2, 1))

## The relationship is proportional

<a href="#sec-analysis" class="quarto-xref">Section 4</a> climbed the model complexity ladder and reported where each rung landed. <a href="#fig-modelcomp" class="quarto-xref">Figure 20</a> collects the ending, held out R² across the five grouped cross validation folds for the two models carried forward. The log target linear model and the random forest perform similarly relative to the variation observed across folds, and neither pulls away from the other. <a href="#tbl-modelmetrics" class="quarto-xref">Table 4</a> adds the single split comparison, including the neural network, which was not cross validated.

In [64]:
cv_means=cv_folds.groupby("model", as_index=False)["r2"].mean().rename(columns={"r2":"mean_r2"})

In [65]:
%%R -i cv_folds -i cv_means -w 8 -h 4 -u in -r 150
ggplot(cv_folds, aes(x=model, y=r2, color=model)) +
  geom_point(size=2.4, alpha=0.7,
             position=position_jitter(width=0.06, height=0, seed=42)) +
  geom_crossbar(data=cv_means, aes(y=mean_r2, ymin=mean_r2, ymax=mean_r2),
                width=0.35, linewidth=0.5, color="grey30") +
  scale_color_manual(values=c("OLS (log target)"="#009E73",
                              "Random forest (log target)"="#CC79A7")) +
  labs(x=NULL, y="Held Out R Squared ($ Scale)") +
  guides(color="none")

In [66]:
metrics_df=pd.DataFrame({
    "Model": ["OLS (log target)","Random forest (log target)","Neural network"],
    "Held out R squared": [ols_log_test_r2, rf_log_test_r2, nn_test_r2],
    "Mean absolute error ($)": [ols_log_test_mae, rf_log_test_mae, nn_test_mae],
})

(GT(metrics_df)
  .fmt_number(columns="Held out R squared", decimals=3)
  .fmt_currency(columns="Mean absolute error ($)", decimals=0)
  .tab_source_note("All models scored on the identical held out counties from the grouped split."))

This is the ladder’s stopping rule working as designed. A random forest can represent any interaction or curvature the data contain, and it finds nothing beyond what the log transformation already captured. The binned comparison in <a href="#fig-binned" class="quarto-xref">Figure 21</a> shows the same conclusion in the units of the outcome. The relationship is not monotonic across these bins: purchasing power rises with routine cognitive share through the lower part of its observed range, peaks near 20 percent, and then declines as the share continues to rise. The forest’s predictions track this shape closely rather than discovering a different pattern of their own, which is further evidence that the flexible model adds little beyond what the log linear specification already captures.

In [67]:
binned_ml=pd.DataFrame({
    "rc": X_test["routine_cognitive_share"].values,
    "Actual": y_test.values,
    "Random Forest": y_pred_rf_log_dollars,
})
binned_ml["bin"]=pd.qcut(binned_ml["rc"], 10, labels=False)
binned_means=binned_ml.groupby("bin")[["rc","Actual","Random Forest"]].mean().reset_index(drop=True)
binned_long=binned_means.melt(id_vars="rc", value_vars=["Actual","Random Forest"],
                              var_name="series", value_name="pp")

In [68]:
%%R -i binned_long -w 8 -h 4.5 -u in -r 150
OBSERVED <- "#252525"; TEAL <- "#1E8E99"
binned_long$rc_pct <- binned_long$rc*100
label_pts <- binned_long[binned_long$rc==max(binned_long$rc),]

ggplot(binned_long, aes(x=rc_pct, y=pp, color=series)) +
  geom_line(aes(linewidth=series)) +
  geom_point(aes(size=series)) +
  geom_text(data=label_pts, aes(label=series), hjust=-0.15, size=3.4, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=c("Actual"=OBSERVED, "Random Forest"=TEAL), guide="none") +
  scale_linewidth_manual(values=c("Actual"=1.0, "Random Forest"=0.7), guide="none") +
  scale_size_manual(values=c("Actual"=1.8, "Random Forest"=1.4), guide="none") +
  scale_x_continuous(labels=function(v) sprintf("%d%%", round(v)),
                      expand=expansion(mult=c(0.02, 0.14))) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=","))) +
  labs(x="Routine Cognitive Group Value (Bin Mean)", y="Mean Purchasing Power")

## What carries the signal

Two measurements say how much the task groups contribute. The first removes them and refits. <a href="#fig-ablation" class="quarto-xref">Figure 22</a> shows what the random forest loses in held out R² when each block of features is taken away. Removing poverty and unemployment costs 0.178, more than four times the 0.042 lost by removing the task groups. The economic controls therefore carry more predictive information, but the task groups still contribute beyond them, which is the question the objection above raised.

In [69]:
ablation_df=pd.DataFrame({
    "model": ["Task groups","Poverty and unemployment"],
    "r2": [0.834, 0.698],
})
ablation_df["full_r2"]=0.876
ablation_df["drop"]=ablation_df["full_r2"]-ablation_df["r2"]

In [70]:
%%R -i ablation_df -w 7 -h 3 -u in -r 150
TEAL <- "#1E8E99"; ORANGE <- "#FF8E32"
ablation_df$model <- factor(ablation_df$model, levels=c("Task groups","Poverty and unemployment"))

ggplot(ablation_df, aes(x=drop, y=model, fill=model)) +
  geom_col(width=0.55) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=4.0, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task groups"=TEAL, "Poverty and unemployment"=ORANGE), guide="none") +
  scale_x_continuous(limits=c(0, 0.20), breaks=seq(0, 0.20, 0.05), expand=c(0,0)) +
  labs(x="Loss in Held Out R Squared", y=NULL)

The second measurement leaves the model intact and destroys the information instead. <a href="#fig-importance" class="quarto-xref">Figure 23</a> ranks what the random forest relies on, using the grouped permutation approach from <a href="#sec-analysis" class="quarto-xref">Section 4</a> so that indicator blocks and the task groups are each scored as a unit. These two interpretability figures are computed on the raw target forest rather than the log target one, since the two score within 0.001 of each other on held out counties and the raw target version reports directly in dollars. Poverty rate carries the largest single drop, consistent with its dominance in the regression. The modeled task group proportions together come next at 0.153, ahead of the year block at 0.128, and state adds almost nothing at 0.008 once everything else is present.

The two measurements disagree in magnitude, and the reason is instructive. Permuting the task groups costs 0.153 while removing them entirely costs 0.042, because a refitted model can lean harder on poverty, unemployment, and time to recover much of what the task groups were carrying. The permutation figure therefore measures how much the fitted model uses the task groups, and the ablation measures how much of that is unique to them. Both say the task groups matter as a set, which is what a set of parts of a whole should do.

In [71]:
pov_idx=list(X_test.columns).index("poverty_rate")
unemp_idx=list(X_test.columns).index("unemployment_rate")

importance_df=pd.DataFrame({
    "feature": ["Poverty Rate","Task Groups (Joint)","Year (Joint)",
                "Unemployment Rate","State (Joint)"],
    "drop": [perm_imp.importances_mean[pov_idx], joint_drop, year_drop,
             perm_imp.importances_mean[unemp_idx], state_drop],
}).sort_values("drop")

In [72]:
%%R -i importance_df -w 8 -h 3.5 -u in -r 150
importance_df$feature <- factor(importance_df$feature, levels=importance_df$feature)

ggplot(importance_df, aes(x=drop, y=feature)) +
  geom_col(width=0.6, fill="#0072B2") +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.2, size=3.4) +
  scale_x_continuous(expand=expansion(mult=c(0, 0.15))) +
  labs(x="Drop in Held Out R Squared When Permuted", y=NULL)

<a href="#fig-pdp" class="quarto-xref">Figure 24</a> completes the picture with the forest’s partial dependence on the three non reference task groups. Each curve shows predicted purchasing power as one group’s value moves across its observed range with all other features held at their values. The curves decline smoothly and near monotonically, with no thresholds or reversals, which is the shape a proportional association implies and the reason the flexible model could not beat the log linear one. Because the four groups are parts of a whole, moving one while holding the others fixed implies a compensating change in the omitted reference group, so these curves describe the model’s behavior rather than a combination any county could occupy.

In [73]:
from sklearn.inspection import partial_dependence

pdp_frames=[]
for term, name in zip(["routine_cognitive_share","routine_manual_share","non_routine_manual_share"],
                      ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    pd_res=partial_dependence(best_rf, X_test, [term], kind="average", grid_resolution=40)
    grid=pd_res["grid_values"][0] if "grid_values" in pd_res else pd_res["values"][0]
    pdp_frames.append(pd.DataFrame({"x":grid,
                                    "y":pd_res["average"][0],
                                    "group":name}))
pdp_df=pd.concat(pdp_frames, ignore_index=True)

In [74]:
%%R -i pdp_df -w 10 -h 3.5 -u in -r 150
ggplot(pdp_df, aes(x=x, y=y)) +
  geom_line(color="#CC79A7", linewidth=0.8) +
  facet_wrap(~group, scales="free_x") +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=","))) +
  labs(x="Task Group Value", y="Predicted Purchasing Power")

## The differences are durable

The final question is whether these associations describe a stable feature of places or a moment in time. Three pieces of evidence say stable. <a href="#fig-taskarea" class="quarto-xref">Figure 25</a> shows the average county task composition over the panel, which moves little across fifteen years apart from the 2009 to 2010 step produced by the Census occupation coding change described in <a href="#sec-data" class="quarto-xref">Section 3</a>.

In [75]:
area_df=(df.groupby("year")[["routine_cognitive_share","routine_manual_share",
                             "non_routine_cognitive_share","non_routine_manual_share"]]
           .mean().reset_index()
           .melt(id_vars="year", var_name="group", value_name="share"))
area_names={"routine_cognitive_share":"Routine Cognitive",
            "routine_manual_share":"Routine Manual",
            "non_routine_cognitive_share":"Non-Routine Cognitive",
            "non_routine_manual_share":"Non-Routine Manual"}
area_df["group"]=area_df["group"].map(area_names)
area_df["period"]=np.where(area_df["year"]<=2019, "pre", "post")

In [76]:
%%R -i area_df -w 9.5 -h 4.3 -u in -r 150
TASKAREA_COLORS <- c(
  "Non-Routine Cognitive" = "#FFAD65",
  "Non-Routine Manual"    = "#993F00",
  "Routine Cognitive"     = "#51C3CC",
  "Routine Manual"        = "#006666"
)
group_levels <- c("Non-Routine Cognitive","Non-Routine Manual","Routine Cognitive","Routine Manual")
area_df$group <- factor(area_df$group, levels=group_levels)

pre_df  <- area_df[area_df$period=="pre",]
post_df <- area_df[area_df$period=="post",]

last_year <- max(post_df$year)
label_df <- post_df[post_df$year==last_year,]
label_df <- label_df[match(group_levels, label_df$group),]
label_df$cum <- cumsum(label_df$share)
label_df$mid <- label_df$cum - label_df$share/2

ggplot(area_df, aes(x=year, y=share, fill=group)) +
  geom_area(data=pre_df, position="stack") +
  geom_area(data=post_df, position="stack") +
  geom_vline(xintercept=2009.5, linetype="dashed", linewidth=0.6, color="grey45") +
  annotate("text", x=2009.5, y=1.04, label="Census coding change", hjust=-0.05, size=3.1, color="grey35") +
  annotate("text", x=2020, y=0.5, label="No ACS estimates\nin 2020", size=3, color="grey45", lineheight=0.9) +
  geom_text(data=label_df, aes(x=last_year+0.4, y=mid, label=group, color=group),
            hjust=0, size=3.15, fontface="bold", show.legend=FALSE) +
  scale_fill_manual(values=TASKAREA_COLORS, guide="none") +
  scale_color_manual(values=TASKAREA_COLORS, guide="none") +
  scale_y_continuous(labels=function(v) sprintf("%.0f%%", v*100), limits=c(0, 1.08), expand=c(0,0)) +
  scale_x_continuous(breaks=unique(area_df$year), expand=expansion(mult=c(0.02, 0.32))) +
  labs(x="Year", y="Mean Group Value Across Counties") +
  theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1),
        plot.margin=margin(5.5, 14, 5.5, 5.5))

Individual counties hold their positions as well, not just the national aggregate. <a href="#tbl-rankcorr" class="quarto-xref">Table 5</a> reports the rank correlation of each group’s county ordering between 2010 and 2023, computed from 2010 onward to step over the coding change. Routine manual and non-routine cognitive hold their orderings most strongly, so the counties with the most of each in 2010 are largely the same counties in 2023. Non-routine manual is more mobile, and routine cognitive reorders the most, which is consistent with it being the one group carrying substantial within county movement in the variance decomposition of <a href="#sec-analysis" class="quarto-xref">Section 4</a>.

In [77]:
from scipy.stats import spearmanr

wide_2010=df[df["year"]==2010].set_index("county_fips")
wide_2023=df[df["year"]==2023].set_index("county_fips")
common=wide_2010.index.intersection(wide_2023.index)

rank_rows=[]
for col, name in area_names.items():
    rho=spearmanr(wide_2010.loc[common, col], wide_2023.loc[common, col]).statistic
    rank_rows.append({"Task group":name, "Rank correlation, 2010 to 2023":rho})
rankcorr_df=pd.DataFrame(rank_rows)

(GT(rankcorr_df)
  .fmt_number(columns="Rank correlation, 2010 to 2023", decimals=2)
  .cols_align(align="right", columns="Rank correlation, 2010 to 2023")
  .tab_source_note("Spearman correlation across counties present in both years."))

Finally, the coefficients themselves hold across time. <a href="#fig-stability" class="quarto-xref">Figure 26</a> refits the log panel regression on the pre pandemic window, 2010 to 2019, and the post pandemic window, 2021 to 2023, the same windows named in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. Reporting these in percent rather than dollars keeps the three windows comparable, since nominal purchasing power rose substantially across the panel and a dollar coefficient in the later window is measured against a larger base. The task group ordering is identical in both windows and in the full panel, and the magnitudes move modestly, so the association is not an artifact of any one period, recession, recovery, or pandemic era.

In [78]:
def fit_window_log(frame):
    Xw=pd.concat([frame[res_cols],
                  pd.get_dummies(frame["year"], prefix="year", drop_first=True).astype(float)], axis=1)
    Xw=sm.add_constant(Xw)
    return sm.OLS(np.log(frame["affordability_salary"]), Xw).fit(
        cov_type="cluster", cov_kwds={"groups": frame["county_fips"]})

windows={"Full panel": reg,
         "2010 to 2019": reg[(reg["year"]>=2010)&(reg["year"]<=2019)],
         "2021 to 2023": reg[reg["year"]>=2021]}

stab_rows=[]
for term, name in zip(group_terms, group_names):
    row={"Task group":name}
    for wname, frame in windows.items():
        b=fit_window_log(frame).params[term]
        row[wname]=(np.exp(b*0.01)-1)*100
    stab_rows.append(row)
stability_df=pd.DataFrame(stab_rows)

stab_long=stability_df.melt(id_vars="Task group", var_name="window", value_name="value")
range_df=stability_df.assign(
    lo=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].min(axis=1),
    hi=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].max(axis=1),
)[["Task group","lo","hi"]]

In [79]:
%%R -i stab_long -i range_df -w 8 -h 3.6 -u in -r 150
TEAL <- "#1E8E99"; ORANGE <- "#EB6834"; BLACK <- "#252525"

group_order <- c("Routine Manual", "Routine Cognitive", "Non-Routine Manual")
stab_long$window <- factor(stab_long$window, levels=c("2010 to 2019", "Full panel", "2021 to 2023"))
stab_long$`Task group` <- factor(stab_long$`Task group`, levels=group_order)
range_df$`Task group` <- factor(range_df$`Task group`, levels=group_order)

window_colors <- c("2010 to 2019"=TEAL, "Full panel"=BLACK, "2021 to 2023"=ORANGE)
window_shapes <- c("2010 to 2019"=16, "Full panel"=18, "2021 to 2023"=16)
window_sizes  <- c("2010 to 2019"=3.2, "Full panel"=4.6, "2021 to 2023"=3.2)

ggplot() +
  geom_segment(data=range_df, aes(x=lo, xend=hi, y=`Task group`, yend=`Task group`),
               color="#C8C8C5", linewidth=1.1) +
  geom_point(data=stab_long, aes(x=value, y=`Task group`, color=window, shape=window, size=window)) +
  scale_color_manual(values=window_colors, name=NULL) +
  scale_shape_manual(values=window_shapes, name=NULL) +
  scale_size_manual(values=window_sizes, guide="none") +
  scale_x_continuous(labels=function(v) sprintf("%.1f%%", v)) +
  labs(x="Percent Change per Point Shifted Out of Non-Routine Cognitive Work", y=NULL) +
  theme(legend.position="bottom")

Together, the stability of the national picture, of the county orderings, and of the coefficients supports reading the headline association as a durable characteristic of local economies. The counties whose work is most routine and most manual entered the study period with less purchasing power, and fifteen years later they largely still do. What this means, what it cannot mean, and where it points next are taken up in <a href="#sec-conclusions" class="quarto-xref">Section 6</a>.

# Conclusions

## Summary of findings

This study asked whether the balance of work in a county is associated with the purchasing power of the people who live there. The answer is yes. Across 848 counties and fifteen years, counties whose work leans toward routine and manual tasks show systematically lower purchasing power than counties whose work leans toward non-routine cognitive tasks, and the association holds after controlling for poverty, unemployment, population, and year. A one percentage point shift toward non-routine manual work and away from non-routine cognitive work is associated with roughly \$846 less in purchasing power at the panel mean. The association is proportional rather than linear in dollars, a reading the diagnostics support and the flexible models are consistent with, since neither the random forest nor the neural network found structure the log linear fit had missed.

In [80]:
%%R -w 8.5 -h 2.6 -u in -r 150
TEAL <- "#1E8E99"; ORANGE <- "#EB6834"; DARK_TEAL <- "#006666"

stat_panel <- function(value, label, color) {
  ggplot() +
    annotate("text", x=0, y=0.62, label=value, size=11, fontface="bold", color=color) +
    annotate("text", x=0, y=0.18, label=label, size=3.9, color="#4A4A45", lineheight=0.95) +
    xlim(-1, 1) + ylim(0, 1) +
    theme_void()
}

p1 <- stat_panel("−$846", "Association per point shifted\ntoward non-routine manual work", TEAL)
p2 <- stat_panel("0.042", "R² the task groups add\nbeyond economic controls alone", ORANGE)
p3 <- stat_panel("15 years", "Span over which the\nassociation held", DARK_TEAL)

p1 | p2 | p3

## Contributions

The introduction named three gaps between prior work and a claim about county purchasing power, and the study closed each one. Where prior outcomes were reported in unadjusted wages, the outcome here is income deflated by local prices, so a dollar means the same thing in every county it describes. Where the unit of analysis was the commuting zone, the unit here is the county, fine enough to separate a metropolitan core from the areas around it. And where prior estimation was linear, the modeling here tested the functional form directly, establishing that the association is proportional and that more flexible models add nothing once that is accounted for. Each gap was closed with public data and methods that any reader can rerun.

The finding is useful to two groups in particular. For policymakers, the task composition of local work marks counties whose purchasing power runs below what poverty and unemployment alone would suggest, which is information the standard distress measures do not carry. For economic development agencies and others deciding where to invest in communities, purchasing power identifies places where nominal income overstates or understates what residents can actually afford, and the balance of work helps explain which counties those are. In both cases the contribution is the same, a measurable, durable characteristic of places that adds signal beyond the measures already in use.

## Limitations

Four limitations bound what the results can support. The findings are associations, not causal effects, since counties were not assigned their task compositions, and any factor tied to both the work a county contains and its purchasing power could account for part of the relationship. The panel covers only counties above 65,000 residents, so rural counties are underrepresented and the estimates describe the more populous three quarters of the population rather than counties in general. The panel regression carries no state effects, so its estimates absorb whatever varies systematically by state; the predictive models, which do include state indicators, suggest that variation is modest once poverty, unemployment, and time are present, but the two specifications are not identical on this point. And a majority of panel rows carry a state level price parity rather than a local one, so the outcome is measured more coarsely in nonmetropolitan counties, and the analysis does not separate the two.

## Ethical considerations

The design also carries ethical boundaries that shape how the results should be read. The unit of analysis is the county, so every claim describes places rather than people, and inferring anything about an individual worker from these results would be an ecological fallacy. The population threshold means the smallest and most rural counties are absent, and any application of these findings to such places would be extrapolation beyond the data. The associational framing is not a hedge but a constraint, since presenting these estimates as causal could misdirect policy toward changing the balance of work when the underlying driver may lie elsewhere. And the study uses public aggregate data only, so no individual’s information enters the analysis at any point.

## Future directions

Two directions follow from the limitations. The coverage gap could be narrowed with satellite imagery. Jean et al. ([2016](#ref-Jean2016)) estimate local economic conditions from daytime and nighttime imagery where survey data are thin. Applied here, that approach could produce purchasing power estimates for the counties below the American Community Survey threshold, which would show whether the geography reported above extends to the places this panel cannot see. It would not extend the association itself, since occupational estimates remain unavailable for those counties, but confirming that the outcome pattern continues below the threshold would tell us the geography is a feature of the country rather than of the sample. The causal question is the harder one, and it would require a source of variation in task composition that does not run through purchasing power, such as plant openings and closures or technology adoption shocks, which is a different study built on different data.

## Conclusion

This study establishes that the balance of work is a durable, measurable characteristic of counties, and that it carries real information about what residents can afford. The counties whose work was most routine and most manual entered the study period with less purchasing power, and fifteen years later they largely still do. Anyone reading only the poverty rate is looking at an incomplete picture of where purchasing power in America is strained, and the task composition of local work is part of what completes it.

# References

Acemoglu, Daron, and David Autor. 2011. “Skills, Tasks and Technologies: Implications for Employment and Earnings.” In *Handbook of Labor Economics*, edited by David Card and Orley Ashenfelter, 4:1043–1171. Elsevier. <https://doi.org/10.1016/S0169-7218(11)02410-5>.

Autor, David H., and David Dorn. 2013. “The Growth of Low-Skill Service Jobs and the Polarization of the US Labor Market.” *American Economic Review* 103 (5): 1553–97. <https://doi.org/10.1257/aer.103.5.1553>.

Autor, David H., Frank Levy, and Richard J. Murnane. 2003. “The Skill Content of Recent Technological Change: An Empirical Exploration.” *The Quarterly Journal of Economics* 118 (4): 1279–1333. <https://doi.org/10.1162/003355303322552801>.

Jean, Neal, Marshall Burke, Michael Xie, W. Matthew Davis, David B. Lobell, and Stefano Ermon. 2016. “Combining Satellite Imagery and Machine Learning to Predict Poverty.” *Science* 353 (6301): 790–94. <https://doi.org/10.1126/science.aaf7894>.

Saad, Lydia. 2023. “More U.S. Workers Fear Technology Making Their Jobs Obsolete.” Gallup. <https://news.gallup.com/poll/510551/workers-fear-technology-making-jobs-obsolete.aspx>.

Smith, Aaron, and Monica Anderson. 2017. “Automation in Everyday Life.” Pew Research Center. <https://www.pewresearch.org/internet/2017/10/04/automation-in-everyday-life/>.

U.S. Census Bureau. 2023. “Vintage 2023 Population Estimates.” <https://www.census.gov/programs-surveys/popest.html>.